In [ ]:
#!pip install -q "numpy>=1.23" "pandas>=1.5" "PyYAML>=6.0" "matplotlib>=3.6" "tqdm>=4.65" numba scipy Pillow gym pyglet

In [ ]:
from __future__ import annotations

import argparse
import collections
import csv
import glob
import json
import math
import cmath
import os
import random
import time
import types
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Protocol, Tuple

import numpy as np
import pandas as pd
import yaml
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt

from numba import njit
from scipy.ndimage import distance_transform_edt as edt

import gym
from gym import error, spaces, utils
from gym.utils import seeding

import pyglet


In [ ]:
from __future__ import annotations



def _to_namespace(d):
    if isinstance(d, dict):
        ns = types.SimpleNamespace()
        for k, v in d.items():
            setattr(ns, k, _to_namespace(v))
        return ns
    if isinstance(d, list):
        return [_to_namespace(v) for v in d]
    return d


def load_config(path: str):
    with open(path, "r") as f:
        raw = yaml.safe_load(f)
    return _to_namespace(raw)


def resolve_device(requested: str) -> str:
    if requested.startswith("cuda") and not torch.cuda.is_available():
        print(f"[config] '{requested}' requested but CUDA is not available -> falling back to CPU.")
        return "cpu"
    return requested

In [ ]:
!git clone https://github.com/f1tenth/f1tenth_racetracks.git
!mkdir -p maps
!cp -r /kaggle/working/f1tenth_racetracks/* ./maps/

In [ ]:
DEFAULT_CONFIG_YAML = r'''

data:
  root: "data/offline"
  maps_root: "maps"
  runs: null
  train_run_frac: 0.95
  subtraj_len: 10
  subsample_stride: 1
  dropna: true
  lidar_dim: 128
  lidar_max_range: 10.0

  state_features:
    - x
    - y
    - steer_angle
    - vel
    - yaw_angle
    - yaw_rate
    - slip_angle

  action_features:
    - steer_cmd
    - speed_cmd

collection:
  map_names: ["Spielberg" , "Austin" , "MoscowRaceway"]
  laps_per_map: 1000
  episodes_per_map: 1000
  run_seconds: 30
  # Cover speeds and recovery states that MPC will visit, rather than fitting
  # a world model only to one nominal raceline controller.
  max_speed_choices: [3.0, 4.0, 5.0, 6.0]
  action_noise: true
  steer_noise_std: 0.06
  speed_noise_std: 0.5
  control_hz: 20
  timestep: 0.01
  wheelbase_m: 0.33
  max_steer_rad: 0.4
  seed: 0
  out_root: "data/offline"
  lidar_num_beams: 20
  lidar_subsample: 20

model:
  z_dim: 16
  hidden_dim: 128
  gru_hidden_dim: 128
  gru_bidirectional: true
  min_std: 1.0e-3
  obs_embed_dim: 128
  # Predict terminal state changes rather than absolute global coordinates.
  # Requires retraining; old checkpoints remain supported for planning.
  # World model decodes final LiDAR scans.
  tawm_predict_delta: false


train:
  # Works for small offline collections too; do not drop the only partial
  # batch, otherwise a one-run dataset yields zero training updates.
  batch_size: 128
  lr_e: 1.0e-4
  lr_m: 1.0e-4
  num_epochs: 300
  steps_per_epoch: 500
  grad_clip_norm: 5.0
  device: "cuda"
  seed: 0
  checkpoint_dir: "checkpoints"
  log_every: 50
  save_every_epochs: 50
  mode: "causal_em"

planning:
  plan_length: 4
  # These settings are fast enough for CPU use while retaining a broad CEM
  # search.  Increase them only after profiling on your machine.
  population_size: 512
  elite_frac: 0.10
  cem_iters: 5
  init_std: 1.0
  # Execute two skills from each 4-skill plan before computing the next plan.
  # This removes the long pause between every individual skill.
  replan_every_skills: 1
  max_skills_per_episode: 300
  max_laps: 1
  waypoint_spacing_m: 3.0
  waypoint_reach_radius_m: 1.5
  waypoint_bonus_reward: 5.0
  progress_reward_weight: 1.0
  # CEM rewards forward arclength, penalizes reverse motion, and rejects
  # predicted trajectories farther than this from the centerline.
  track_half_width_m: 1.5
  off_track_penalty: 20.0
  backward_penalty: 2.0
  max_progress_per_skill_m: 12.0
  # Occupancy-map collisions are terminal in imagined rollouts.  The penalty
  # must remain larger than all possible horizon progress.
  collision_penalty: 1000.0
  device: "cuda"
  forward_reward_weight: 1.0
  symmetry_penalty_weight: 0.2
  lidar_collision_penalty: 100.0
  safe_distance_m: 0.35
  forward_beam_fraction: 0.30
  reward: "lidar_structural"

mpc:
  horizon: 15
  population_size: 512
  cem_iters: 5
  elite_frac: 0.15
  init_steer_std: 0.15
  init_speed_std: 1.5
  dt: 0.05
  max_seconds: 60.0

f1tenth_env:
  render: true
  control_hz: 20
  timestep: 0.01
  max_steer_rad: 0.4
  max_speed_mps: 8.0
  # Must match the policy/checkpoint LiDAR encoder width.  The planner uses
  # the checkpoint value automatically, so this is documentation only.
  lidar_dim: 20

evaluation:
  num_eval_windows: 500
  num_planning_runs: 5

'''

os.makedirs("configs", exist_ok=True)
with open("configs/default.yaml", "w") as f:
    f.write(DEFAULT_CONFIG_YAML)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
class Logger:
    def __init__(self, log_every: int = 20):
        self.log_every = log_every
        self._history = collections.defaultdict(list)
        self._t0 = time.time()

    def log(self, step: int, **scalars):
        for k, v in scalars.items():
            self._history[k].append(float(v))
        if step % self.log_every == 0:
            elapsed = time.time() - self._t0
            msg = f"[step {step:>7d} | {elapsed:8.1f}s] "
            parts = []
            for k, vals in self._history.items():
                recent = vals[-self.log_every:]
                parts.append(f"{k}={sum(recent) / len(recent):.4f}")
            print(msg + "  ".join(parts))

    def history(self):
        return dict(self._history)

In [ ]:
@dataclass
class Normalizer:
    mean: np.ndarray
    std: np.ndarray
    feature_names: List[str] = field(default_factory=list)
    eps: float = 1e-6

    @classmethod
    def fit(cls, data: np.ndarray, feature_names: List[str] | None = None) -> "Normalizer":
        """data: [N, D]"""
        mean = data.mean(axis=0)
        std = data.std(axis=0)
        std = np.where(std < 1e-6, 1.0, std)
        return cls(mean=mean, std=std, feature_names=feature_names or [])

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean) / (self.std + self.eps)

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * (self.std + self.eps) + self.mean

    def to_dict(self):
        return {
            "mean": self.mean.tolist(),
            "std": self.std.tolist(),
            "feature_names": self.feature_names,
        }

    @classmethod
    def from_dict(cls, d):
        return cls(
            mean=np.array(d["mean"], dtype=np.float32),
            std=np.array(d["std"], dtype=np.float32),
            feature_names=d.get("feature_names", []),
        )

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def load(cls, path: str) -> "Normalizer":
        with open(path, "r") as f:
            return cls.from_dict(json.load(f))

In [ ]:
def _read_centerline_csv(path: str) -> pd.DataFrame:
    with open(path, "r") as f:
        first_line = f.readline()

    if ";" in first_line:
        header = [c.strip().lstrip("#").strip() for c in first_line.split(";")]
        df = pd.read_csv(path, sep=";", comment=None, names=header, skiprows=1)
    else:
        df = pd.read_csv(path)
    df.columns = [c.strip().lstrip("#").strip() for c in df.columns]
    return df


@dataclass
class Centerline:

    points: np.ndarray          # [N, 2] (x, y) centerline points, in track order
    track_half_width: Optional[np.ndarray] = None  # [N] mean of left/right widths, if available

    def __post_init__(self):
        seg = np.diff(self.points, axis=0, append=self.points[:1])
        seg_len = np.linalg.norm(seg, axis=1)
        self.cum_dist = np.concatenate([[0.0], np.cumsum(seg_len)])[:-1]  # [N]
        self.total_length = float(np.sum(seg_len))
        self.num_points = self.points.shape[0]

    @classmethod
    def from_csv(cls, path: str, x_col: str = "x_m", y_col: str = "y_m",
                 width_left_col: str = "w_tr_left_m", width_right_col: str = "w_tr_right_m"
                 ) -> "Centerline":
        df = _read_centerline_csv(path)
        if x_col not in df.columns or y_col not in df.columns:
            raise ValueError(
                f"Centerline file '{path}' is missing '{x_col}'/'{y_col}' columns; "
                f"found columns: {list(df.columns)}. If your map uses different "
                f"column names, update utils/track.py's x_col/y_col arguments."
            )
        points = df[[x_col, y_col]].to_numpy(dtype=np.float32)

        half_width = None
        if width_left_col in df.columns and width_right_col in df.columns:
            half_width = (
                (df[width_left_col].to_numpy(dtype=np.float32)
                 + df[width_right_col].to_numpy(dtype=np.float32)) / 2.0
            )
        return cls(points=points, track_half_width=half_width)

    def subsample_waypoints(self, spacing_m: float) -> np.ndarray:
        """Returns a [M, 2] subset of centerline points spaced roughly
        `spacing_m` apart along the track, used as planning-time
        'checkpoints' (the F1TENTH analogue of the drone track's gates)."""
        if spacing_m <= 0:
            return self.points.copy()
        targets = np.arange(0.0, self.total_length, spacing_m)
        idx = np.searchsorted(self.cum_dist, targets, side="left")
        idx = np.clip(idx, 0, self.num_points - 1)
        return self.points[idx]

    def nearest_index(self, xy: np.ndarray) -> int:
        d2 = np.sum((self.points - xy[None, :]) ** 2, axis=1)
        return int(np.argmin(d2))

    def progress_m(self, xy: np.ndarray) -> float:
        """Approximate arc-length progress (in meters, [0, total_length))
        of the closest centerline point to `xy`."""
        return float(self.cum_dist[self.nearest_index(xy)])

    def lateral_error_m(self, xy: np.ndarray) -> float:
        """Distance from `xy` to the nearest centerline point -- used as a
        cheap off-track proxy when per-point track widths aren't available."""
        idx = self.nearest_index(xy)
        return float(np.linalg.norm(xy - self.points[idx]))


def discover_map(maps_root: str, map_name: str):
    """Resolves the (image, yaml, centerline_csv) triple for a map, following
    the f1tenth_racetracks layout:

        maps_root/<map_name>/<map_name>_map.png
        maps_root/<map_name>/<map_name>_map.yaml
        maps_root/<map_name>/<map_name>_centerline.csv

    Returns (map_yaml_path, map_ext, centerline_csv_path). Raises with a
    helpful message (pointing at f1tenth_racetracks) if the map isn't found.
    """
    map_dir = os.path.join(maps_root, map_name)
    map_yaml = os.path.join(map_dir, f"{map_name}_map.yaml")
    centerline_csv = os.path.join(map_dir, f"{map_name}_centerline.csv")

    if not os.path.exists(map_yaml):
        raise FileNotFoundError(
            f"Could not find '{map_yaml}'. Download F1TENTH tracks (e.g. from "
            f"https://github.com/f1tenth/f1tenth_racetracks) and place them under "
            f"'{maps_root}/<map_name>/' following the '<map_name>_map.yaml' / "
            f"'<map_name>_map.png' / '<map_name>_centerline.csv' naming convention, "
            f"or point `data.maps_root` / `data.map_names` in your config at wherever "
            f"you've put them."
        )
    for ext in (".png", ".pgm"):
        candidate = os.path.join(map_dir, f"{map_name}_map{ext}")
        if os.path.exists(candidate):
            map_ext = ext
            break
    else:
        raise FileNotFoundError(f"No '{map_name}_map.png' or '.pgm' found in {map_dir}")

    if not os.path.exists(centerline_csv):
        raise FileNotFoundError(
            f"Could not find '{centerline_csv}'. This is only used at "
            f"planning/reward time (never for training) -- f1tenth_racetracks "
            f"ships a '<map>_centerline.csv' for every track."
        )
    return map_yaml, map_ext, centerline_csv

In [ ]:
from __future__ import annotations

from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal




def mlp(in_dim: int, hidden_dim: int, out_dim: int, num_hidden_layers: int = 1) -> nn.Sequential:
    layers = [nn.Linear(in_dim, hidden_dim), nn.ReLU()]
    for _ in range(num_hidden_layers - 1):
        layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
    layers += [nn.Linear(hidden_dim, out_dim)]
    return nn.Sequential(*layers)


class GaussianHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, min_std: float = 1e-3):
        super().__init__()
        self.mean_net = mlp(in_dim, hidden_dim, out_dim, num_hidden_layers=2)
        self.std_net = mlp(in_dim, hidden_dim, out_dim, num_hidden_layers=2)
        self.min_std = min_std
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        mu = self.mean_net(x)
        sigma = F.softplus(self.std_net(x)) + self.min_std
        return mu, sigma

    def dist(self, x: torch.Tensor) -> Normal:
        mu, sigma = self.forward(x)
        return Normal(mu, sigma)


class SkillPosterior(nn.Module):

    def __init__(self, obs_embed_dim: int, action_dim: int, z_dim: int,
                 hidden_dim: int = 256, gru_hidden_dim: int = 128,
                 bidirectional: bool = True, min_std: float = 1e-3):
        super().__init__()
        self.obs_embed_dim = obs_embed_dim
        gru_in_dim = obs_embed_dim + action_dim
        self.gru = nn.GRU(
            input_size=gru_in_dim,
            hidden_size=gru_hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional,
        )
        gru_out_dim = gru_hidden_dim * (2 if bidirectional else 1)
        self.head = GaussianHead(gru_out_dim, hidden_dim, z_dim, min_std=min_std)

    def forward(self, lidar_emb_seq: torch.Tensor, actions_seq: torch.Tensor):
        gru_in = torch.cat([lidar_emb_seq, actions_seq], dim=-1)
        gru_out, _ = self.gru(gru_in)                       
        pooled = gru_out.mean(dim=1)                        
        return self.head(pooled)

    def dist(self, lidar_emb_seq: torch.Tensor, actions_seq: torch.Tensor) -> Normal:
        mu, sigma = self.forward(lidar_emb_seq, actions_seq)
        return Normal(mu, sigma)


class SkillPrior(nn.Module):

    def __init__(self, obs_embed_dim: int, z_dim: int, hidden_dim: int = 256, min_std: float = 1e-3):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(obs_embed_dim, hidden_dim), nn.ReLU())
        self.head = GaussianHead(hidden_dim, hidden_dim, z_dim, min_std=min_std)

    def forward(self, o0_embed: torch.Tensor):
        h = self.trunk(o0_embed)
        return self.head(h)


class LidarEncoder(nn.Module):
    def __init__(self, lidar_dim: int, embed_dim: int, hidden_dim: int = 256,
                 max_range: float = 10.0):
        super().__init__()
        self.max_range = max_range
        self.net = mlp(lidar_dim, hidden_dim, embed_dim, num_hidden_layers=2)

    def forward(self, o: torch.Tensor) -> torch.Tensor:
        o_norm = torch.clamp(o, min=0.0, max=self.max_range) / self.max_range
        return F.relu(self.net(o_norm))


class LowLevelPolicy(nn.Module):
    def __init__(self, obs_embed_dim: int, z_dim: int, action_dim: int,
                 hidden_dim: int = 256, min_std: float = 1e-3):
        super().__init__()
        self.gru = nn.GRU(input_size=obs_embed_dim + z_dim, hidden_size=hidden_dim,
                          batch_first=True)
        self.head = GaussianHead(hidden_dim, hidden_dim, action_dim, min_std=min_std)

    def forward(self, lidar_embed_seq: torch.Tensor, z_seq: torch.Tensor):
        single_step = lidar_embed_seq.ndim == 2
        if single_step:
            lidar_embed_seq = lidar_embed_seq.unsqueeze(1)
            z_seq = z_seq.unsqueeze(1)
        h_seq, _ = self.gru(torch.cat([lidar_embed_seq, z_seq], dim=-1))
        mu, sigma = self.head(h_seq)
        if single_step:
            return mu[:, 0], sigma[:, 0]
        return mu, sigma

    def sequence(self, lidar_embed_seq: torch.Tensor, z: torch.Tensor):
        if lidar_embed_seq.ndim != 3 or z.ndim != 2:
            raise ValueError("expected lidar embeddings (batch, horizon, embed_dim) and z (batch, z_dim)")
        z_seq = z.unsqueeze(1).expand(-1, lidar_embed_seq.shape[1], -1)
        return self.dist(lidar_embed_seq, z_seq)

    def step(self, lidar_embed: torch.Tensor, z: torch.Tensor, hidden=None):
        inp = torch.cat([lidar_embed, z], dim=-1).unsqueeze(1)
        h, hidden = self.gru(inp, hidden)
        mu, sigma = self.head(h[:, 0])
        return Normal(mu, sigma), hidden

    def dist(self, lidar_embed_seq: torch.Tensor, z_seq: torch.Tensor) -> Normal:
        mu, sigma = self.forward(lidar_embed_seq, z_seq)
        return Normal(mu, sigma)


class TemporallyAbstractWorldModel(nn.Module):
    def __init__(self, lidar_dim: int, obs_embed_dim: int, z_dim: int, hidden_dim: int = 256,
                 min_std: float = 1e-3):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_embed_dim + z_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.head = GaussianHead(hidden_dim, hidden_dim, lidar_dim, min_std=min_std)

    def forward(self, o0_embed: torch.Tensor, z: torch.Tensor):
        h = self.trunk(torch.cat([o0_embed, z], dim=-1))
        return self.head(h)

    def dist(self, o0_embed: torch.Tensor, z: torch.Tensor) -> Normal:
        mu, sigma = self.forward(o0_embed, z)
        return Normal(mu, sigma)


In [ ]:
from __future__ import annotations

from typing import Dict

import torch
import torch.nn as nn
from torch.distributions import kl_divergence

from models.networks import (
    GaussianHead,
    LidarEncoder,
    SkillPosterior,
    SkillPrior,
    LowLevelPolicy,
    TemporallyAbstractWorldModel,
)

class OPOSMSkillModel(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, z_dim: int, lidar_dim: int,
                 hidden_dim: int = 256, gru_hidden_dim: int = 128,
                 gru_bidirectional: bool = True, min_std: float = 1e-3,
                 obs_embed_dim: int = 128, lidar_max_range: float = 10.0,
                 tawm_predict_delta: bool = False):
        super().__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.z_dim = z_dim
        self.lidar_dim = lidar_dim
        self.obs_embed_dim = obs_embed_dim
        self.tawm_predict_delta = tawm_predict_delta

        self.skill_posterior = SkillPosterior(
            obs_embed_dim, action_dim, z_dim, hidden_dim, gru_hidden_dim,
            gru_bidirectional, min_std=min_std,
        )
        self.lidar_encoder = LidarEncoder(lidar_dim, obs_embed_dim, hidden_dim, max_range=lidar_max_range)
        self.skill_prior = SkillPrior(obs_embed_dim, z_dim, hidden_dim, min_std=min_std)
        self.low_level_policy = LowLevelPolicy(obs_embed_dim, z_dim, action_dim, hidden_dim, min_std=min_std)
        self.tawm = TemporallyAbstractWorldModel(
            lidar_dim, obs_embed_dim, z_dim, hidden_dim, min_std=min_std,
        )
        self.lidar_max_range = lidar_max_range
    def _sample_z(self, lidar_seq, actions_seq):
        """Reparameterized sample from q_phi(z | tau_T)."""
        lidar_emb_seq = self.lidar_encoder(lidar_seq)
        q_dist = self.skill_posterior.dist(lidar_emb_seq, actions_seq)
        z = q_dist.rsample()
        return z, q_dist

    def _low_level_log_prob(self, lidar_seq: torch.Tensor, actions_seq: torch.Tensor,
                             z: torch.Tensor) -> torch.Tensor:
        B, T, _ = actions_seq.shape
        z_rep = z.unsqueeze(1).expand(-1, T, -1)
        lidar_emb_seq = self.lidar_encoder(lidar_seq)
        policy_dist = self.low_level_policy.dist(lidar_emb_seq, z_rep)
        log_prob = policy_dist.log_prob(actions_seq)
        return log_prob.sum(dim=(1, 2))

    def _tawm_log_prob(self, o0: torch.Tensor, oT: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        o0_embed = self.lidar_encoder(o0)
        tawm_dist = self.tawm.dist(o0_embed, z)
        oT_normalized = torch.clamp(oT, min=0.0, max=self.lidar_max_range) / self.lidar_max_range
        log_prob = tawm_dist.log_prob(oT_normalized)
        return log_prob.sum(dim=-1)                          

    def _prior_log_prob(self, o0: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        o0_embed = self.lidar_encoder(o0)
        mu0, sigma0 = self.skill_prior(o0_embed)
        prior = torch.distributions.Normal(mu0, sigma0)
        return prior.log_prob(z).sum(dim=-1)                  

    def elbo_terms(self, batch: Dict[str, torch.Tensor], detach_z: bool = False
                    ) -> Dict[str, torch.Tensor]:

        actions_seq, lidar_seq = batch["actions_seq"], batch["lidar_seq"]
        # LidarEncoder performs the single required range normalization.  The
        # live F1TENTH policy receives raw metre ranges, so training must too.
        z, q_dist = self._sample_z(lidar_seq, actions_seq)
        z_for_networks = z.detach() if detach_z else z

        log_pi = self._low_level_log_prob(lidar_seq, actions_seq, z_for_networks)
        log_psi = self._tawm_log_prob(lidar_seq[:, 0], batch["lidar_T"], z_for_networks)

        o0_embed = self.lidar_encoder(lidar_seq[:, 0])
        mu0, sigma0 = self.skill_prior(o0_embed)
        prior_dist = torch.distributions.Normal(mu0, sigma0)
        kl = kl_divergence(q_dist, prior_dist).sum(dim=-1)              

        elbo = log_pi + log_psi - kl
        return {
            "z": z, "q_dist": q_dist, "prior_dist": prior_dist,
            "log_pi": log_pi, "log_psi": log_psi, "kl": kl, "elbo": elbo,
        }

    def m_loss(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        terms = self.elbo_terms(batch, detach_z=True)
        loss = -terms["elbo"].mean()
        return {
            "loss": loss,
            "elbo": terms["elbo"].mean().detach(),
            "log_pi": terms["log_pi"].mean().detach(),
            "log_psi": terms["log_psi"].mean().detach(),
            "kl": terms["kl"].mean().detach(),
        }

    def e_loss_causal(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:

        s0, states_seq, actions_seq, lidar_seq = (
            batch["s0"], batch["states_seq"], batch["actions_seq"], batch["lidar_seq"]
        )
        z, q_dist = self._sample_z(lidar_seq, actions_seq)

        log_q = q_dist.log_prob(z).sum(dim=-1)
        log_pi = self._low_level_log_prob(lidar_seq, actions_seq, z)
        log_prior = self._prior_log_prob(lidar_seq[:, 0], z)

        kl_to_true_posterior = log_q - log_pi - log_prior
        loss = kl_to_true_posterior.mean()
        return {"loss": loss, "kl_to_true_posterior": loss.detach()}

    def naive_vi_e_loss(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        terms = self.elbo_terms(batch)
        loss = -terms["elbo"].mean()
        return {"loss": loss, "elbo": terms["elbo"].mean().detach()}


    @torch.no_grad()
    def prior_mean_std(self, o0: torch.Tensor):
        return self.skill_prior(self.lidar_encoder(o0))

    @torch.no_grad()
    def predict_terminal_lidar(self, o0: torch.Tensor, z: torch.Tensor, sample: bool = False):
        mu, sigma = self.tawm(self.lidar_encoder(o0), z)
        if sample:
            mu = mu + sigma * torch.randn_like(sigma)
        return mu.clamp(0.0, 1.0) * self.lidar_max_range

    @torch.no_grad()
    def act(self, o: torch.Tensor, z: torch.Tensor, sample: bool = True):
        o_embed = self.lidar_encoder(o)
        mu, sigma = self.low_level_policy(o_embed, z)
        if sample:
            return mu + sigma * torch.randn_like(sigma)
        return mu


In [ ]:
class TAWMRolloutEnvironment:
    def __init__(self, *args, **kwargs):
        raise NotImplementedError(
            "The world model predicts physical terminal state from the current LiDAR embedding and z,"
            "but does not predict future LiDAR. Use F1TenthEnvironment for closed-loop planning."
        )


class KinematicSimEnvironment:
    def __init__(self, *args, **kwargs):
        raise NotImplementedError(
            "This environment cannot provide LiDAR required to sample the skill prior; use F1TenthEnvironment."
        )


In [ ]:
@njit(cache=True)
def perpendicular(pt):
    """
    Return a 2-vector's perpendicular vector

    Args:
        pt (np.ndarray, (2,)): input vector

    Returns:
        pt (np.ndarray, (2,)): perpendicular vector
    """
    temp = pt[0]
    pt[0] = pt[1]
    pt[1] = -1*temp
    return pt


@njit(cache=True)
def tripleProduct(a, b, c):
    """
    Return triple product of three vectors

    Args:
        a, b, c (np.ndarray, (2,)): input vectors

    Returns:
        (np.ndarray, (2,)): triple product
    """
    ac = a.dot(c)
    bc = b.dot(c)
    return b*ac - a*bc


@njit(cache=True)
def avgPoint(vertices):
    """
    Return the average point of multiple vertices

    Args:
        vertices (np.ndarray, (n, 2)): the vertices we want to find avg on

    Returns:
        avg (np.ndarray, (2,)): average point of the vertices
    """
    return np.sum(vertices, axis=0)/vertices.shape[0]


@njit(cache=True)
def indexOfFurthestPoint(vertices, d):
    """
    Return the index of the vertex furthest away along a direction in the list of vertices

    Args:
        vertices (np.ndarray, (n, 2)): the vertices we want to find avg on

    Returns:
        idx (int): index of the furthest point
    """
    return np.argmax(vertices.dot(d))


@njit(cache=True)
def support(vertices1, vertices2, d):
    """
    Minkowski sum support function for GJK

    Args:
        vertices1 (np.ndarray, (n, 2)): vertices of the first body
        vertices2 (np.ndarray, (n, 2)): vertices of the second body
        d (np.ndarray, (2, )): direction to find the support along

    Returns:
        support (np.ndarray, (n, 2)): Minkowski sum
    """
    i = indexOfFurthestPoint(vertices1, d)
    j = indexOfFurthestPoint(vertices2, -d)
    return vertices1[i] - vertices2[j]


@njit(cache=True)
def collision(vertices1, vertices2):
    """
    GJK test to see whether two bodies overlap

    Args:
        vertices1 (np.ndarray, (n, 2)): vertices of the first body
        vertices2 (np.ndarray, (n, 2)): vertices of the second body

    Returns:
        overlap (boolean): True if two bodies collide
    """
    index = 0
    simplex = np.empty((3, 2))

    position1 = avgPoint(vertices1)
    position2 = avgPoint(vertices2)

    d = position1 - position2

    if d[0] == 0 and d[1] == 0:
        d[0] = 1.0

    a = support(vertices1, vertices2, d)
    simplex[index, :] = a

    if d.dot(a) <= 0:
        return False

    d = -a

    iter_count = 0
    while iter_count < 1e3:
        a = support(vertices1, vertices2, d)
        index += 1
        simplex[index, :] = a
        if d.dot(a) <= 0:
            return False

        ao = -a

        if index < 2:
            b = simplex[0, :]
            ab = b-a
            d = tripleProduct(ab, ao, ab)
            if np.linalg.norm(d) < 1e-10:
                d = perpendicular(ab)
            continue

        b = simplex[1, :]
        c = simplex[0, :]
        ab = b-a
        ac = c-a

        acperp = tripleProduct(ab, ac, ac)

        if acperp.dot(ao) >= 0:
            d = acperp
        else:
            abperp = tripleProduct(ac, ab, ab)
            if abperp.dot(ao) < 0:
                return True
            simplex[0, :] = simplex[1, :]
            d = abperp

        simplex[1, :] = simplex[2, :]
        index -= 1

        iter_count += 1
    return False

@njit(cache=True)
def collision_multiple(vertices):
    """
    Check pair-wise collisions for all provided vertices

    Args:
        vertices (np.ndarray (num_bodies, 4, 2)): all vertices for checking pair-wise collision

    Returns:
        collisions (np.ndarray (num_vertices, )): whether each body is in collision
        collision_idx (np.ndarray (num_vertices, )): which index of other body is each index's body is in collision, -1 if not in collision
    """
    collisions = np.zeros((vertices.shape[0], ))
    collision_idx = -1 * np.ones((vertices.shape[0], ))
    # looping over all pairs
    for i in range(vertices.shape[0]-1):
        for j in range(i+1, vertices.shape[0]):
            # check collision
            vi = np.ascontiguousarray(vertices[i, :, :])
            vj = np.ascontiguousarray(vertices[j, :, :])
            ij_collision = collision(vi, vj)
            # fill in results
            if ij_collision:
                collisions[i] = 1.
                collisions[j] = 1.
                collision_idx[i] = j
                collision_idx[j] = i

    return collisions, collision_idx

"""
Utility functions for getting vertices by pose and shape
"""

@njit(cache=True)
def get_trmtx(pose):
    """
    Get transformation matrix of vehicle frame -> global frame

    Args:
        pose (np.ndarray (3, )): current pose of the vehicle

    return:
        H (np.ndarray (4, 4)): transformation matrix
    """
    x = pose[0]
    y = pose[1]
    th = pose[2]
    cos = np.cos(th)
    sin = np.sin(th)
    H = np.array([[cos, -sin, 0., x], [sin, cos, 0., y], [0., 0., 1., 0.], [0., 0., 0., 1.]])
    return H

@njit(cache=True)
def get_vertices(pose, length, width):
    """
    Utility function to return vertices of the car body given pose and size

    Args:
        pose (np.ndarray, (3, )): current world coordinate pose of the vehicle
        length (float): car length
        width (float): car width

    Returns:
        vertices (np.ndarray, (4, 2)): corner vertices of the vehicle body
    """
    H = get_trmtx(pose)
    rl = H.dot(np.asarray([[-length/2],[width/2],[0.], [1.]])).flatten()
    rr = H.dot(np.asarray([[-length/2],[-width/2],[0.], [1.]])).flatten()
    fl = H.dot(np.asarray([[length/2],[width/2],[0.], [1.]])).flatten()
    fr = H.dot(np.asarray([[length/2],[-width/2],[0.], [1.]])).flatten()
    rl = rl/rl[3]
    rr = rr/rr[3]
    fl = fl/fl[3]
    fr = fr/fr[3]
    vertices = np.asarray([[rl[0], rl[1]], [rr[0], rr[1]], [fr[0], fr[1]], [fl[0], fl[1]]])
    return vertices

In [ ]:
@njit(cache=True)
def accl_constraints(vel, accl, v_switch, a_max, v_min, v_max):
    """
    Acceleration constraints, adjusts the acceleration based on constraints

        Args:
            vel (float): current velocity of the vehicle
            accl (float): unconstraint desired acceleration
            v_switch (float): switching velocity (velocity at which the acceleration is no longer able to create wheel spin)
            a_max (float): maximum allowed acceleration
            v_min (float): minimum allowed velocity
            v_max (float): maximum allowed velocity

        Returns:
            accl (float): adjusted acceleration
    """

    # positive accl limit
    if vel > v_switch:
        pos_limit = a_max*v_switch/vel
    else:
        pos_limit = a_max

    # accl limit reached?
    if (vel <= v_min and accl <= 0) or (vel >= v_max and accl >= 0):
        accl = 0.
    elif accl <= -a_max:
        accl = -a_max
    elif accl >= pos_limit:
        accl = pos_limit

    return accl

@njit(cache=True)
def steering_constraint(steering_angle, steering_velocity, s_min, s_max, sv_min, sv_max):
    """
    Steering constraints, adjusts the steering velocity based on constraints

        Args:
            steering_angle (float): current steering_angle of the vehicle
            steering_velocity (float): unconstraint desired steering_velocity
            s_min (float): minimum steering angle
            s_max (float): maximum steering angle
            sv_min (float): minimum steering velocity
            sv_max (float): maximum steering velocity

        Returns:
            steering_velocity (float): adjusted steering velocity
    """

    # constraint steering velocity
    if (steering_angle <= s_min and steering_velocity <= 0) or (steering_angle >= s_max and steering_velocity >= 0):
        steering_velocity = 0.
    elif steering_velocity <= sv_min:
        steering_velocity = sv_min
    elif steering_velocity >= sv_max:
        steering_velocity = sv_max

    return steering_velocity


@njit(cache=True)
def vehicle_dynamics_ks(x, u_init, mu, C_Sf, C_Sr, lf, lr, h, m, I, s_min, s_max, sv_min, sv_max, v_switch, a_max, v_min, v_max):
    """
    Single Track Kinematic Vehicle Dynamics.

        Args:
            x (numpy.ndarray (3, )): vehicle state vector (x1, x2, x3, x4, x5)
                x1: x position in global coordinates
                x2: y position in global coordinates
                x3: steering angle of front wheels
                x4: velocity in x direction
                x5: yaw angle
            u (numpy.ndarray (2, )): control input vector (u1, u2)
                u1: steering angle velocity of front wheels
                u2: longitudinal acceleration

        Returns:
            f (numpy.ndarray): right hand side of differential equations
    """
    # wheelbase
    lwb = lf + lr

    # constraints
    u = np.array([steering_constraint(x[2], u_init[0], s_min, s_max, sv_min, sv_max), accl_constraints(x[3], u_init[1], v_switch, a_max, v_min, v_max)])

    # system dynamics
    f = np.array([x[3]*np.cos(x[4]),
         x[3]*np.sin(x[4]),
         u[0],
         u[1],
         x[3]/lwb*np.tan(x[2])])
    return f

@njit(cache=True)
def vehicle_dynamics_st(x, u_init, mu, C_Sf, C_Sr, lf, lr, h, m, I, s_min, s_max, sv_min, sv_max, v_switch, a_max, v_min, v_max):
    """
    Single Track Dynamic Vehicle Dynamics.

        Args:
            x (numpy.ndarray (3, )): vehicle state vector (x1, x2, x3, x4, x5, x6, x7)
                x1: x position in global coordinates
                x2: y position in global coordinates
                x3: steering angle of front wheels
                x4: velocity in x direction
                x5: yaw angle
                x6: yaw rate
                x7: slip angle at vehicle center
            u (numpy.ndarray (2, )): control input vector (u1, u2)
                u1: steering angle velocity of front wheels
                u2: longitudinal acceleration

        Returns:
            f (numpy.ndarray): right hand side of differential equations
    """

    # gravity constant m/s^2
    g = 9.81

    # constraints
    u = np.array([steering_constraint(x[2], u_init[0], s_min, s_max, sv_min, sv_max), accl_constraints(x[3], u_init[1], v_switch, a_max, v_min, v_max)])

    # switch to kinematic model for small velocities
    if abs(x[3]) < 0.5:
        # wheelbase
        lwb = lf + lr

        # system dynamics
        x_ks = x[0:5]
        f_ks = vehicle_dynamics_ks(x_ks, u, mu, C_Sf, C_Sr, lf, lr, h, m, I, s_min, s_max, sv_min, sv_max, v_switch, a_max, v_min, v_max)
        f = np.hstack((f_ks, np.array([u[1]/lwb*np.tan(x[2])+x[3]/(lwb*np.cos(x[2])**2)*u[0],
        0])))

    else:
        # system dynamics
        f = np.array([x[3]*np.cos(x[6] + x[4]),
            x[3]*np.sin(x[6] + x[4]),
            u[0],
            u[1],
            x[5],
            -mu*m/(x[3]*I*(lr+lf))*(lf**2*C_Sf*(g*lr-u[1]*h) + lr**2*C_Sr*(g*lf + u[1]*h))*x[5] \
                +mu*m/(I*(lr+lf))*(lr*C_Sr*(g*lf + u[1]*h) - lf*C_Sf*(g*lr - u[1]*h))*x[6] \
                +mu*m/(I*(lr+lf))*lf*C_Sf*(g*lr - u[1]*h)*x[2],
            (mu/(x[3]**2*(lr+lf))*(C_Sr*(g*lf + u[1]*h)*lr - C_Sf*(g*lr - u[1]*h)*lf)-1)*x[5] \
                -mu/(x[3]*(lr+lf))*(C_Sr*(g*lf + u[1]*h) + C_Sf*(g*lr-u[1]*h))*x[6] \
                +mu/(x[3]*(lr+lf))*(C_Sf*(g*lr-u[1]*h))*x[2]])

    return f

@njit(cache=True)
def pid(speed, steer, current_speed, current_steer, max_sv, max_a, max_v, min_v):
    """
    Basic controller for speed/steer -> accl./steer vel.

        Args:
            speed (float): desired input speed
            steer (float): desired input steering angle

        Returns:
            accl (float): desired input acceleration
            sv (float): desired input steering velocity
    """
    # steering
    steer_diff = steer - current_steer
    if np.fabs(steer_diff) > 1e-4:
        sv = (steer_diff / np.fabs(steer_diff)) * max_sv
    else:
        sv = 0.0

    # accl
    vel_diff = speed - current_speed
    # currently forward
    if current_speed > 0.:
        if (vel_diff > 0):
            # accelerate
            kp = 10.0 * max_a / max_v
            accl = kp * vel_diff
        else:
            # braking
            kp = 10.0 * max_a / (-min_v)
            accl = kp * vel_diff
    # currently backwards
    else:
        if (vel_diff > 0):
            # braking
            kp = 2.0 * max_a / max_v
            accl = kp * vel_diff
        else:
            # accelerating
            kp = 2.0 * max_a / (-min_v)
            accl = kp * vel_diff

    return accl, sv

def func_KS(x, t, u, mu, C_Sf, C_Sr, lf, lr, h, m, I, s_min, s_max, sv_min, sv_max, v_switch, a_max, v_min, v_max):
    f = vehicle_dynamics_ks(x, u, mu, C_Sf, C_Sr, lf, lr, h, m, I, s_min, s_max, sv_min, sv_max, v_switch, a_max, v_min, v_max)
    return f

def func_ST(x, t, u, mu, C_Sf, C_Sr, lf, lr, h, m, I, s_min, s_max, sv_min, sv_max, v_switch, a_max, v_min, v_max):
    f = vehicle_dynamics_st(x, u, mu, C_Sf, C_Sr, lf, lr, h, m, I, s_min, s_max, sv_min, sv_max, v_switch, a_max, v_min, v_max)
    return f

In [ ]:
def get_dt(bitmap, resolution):
    """
    Distance transformation, returns the distance matrix from the input bitmap.
    Uses scipy.ndimage, cannot be JITted.

        Args:
            bitmap (numpy.ndarray, (n, m)): input binary bitmap of the environment, where 0 is obstacles, and 255 (or anything > 0) is freespace
            resolution (float): resolution of the input bitmap (m/cell)

        Returns:
            dt (numpy.ndarray, (n, m)): output distance matrix, where each cell has the corresponding distance (in meters) to the closest obstacle
    """
    dt = resolution * edt(bitmap)
    return dt

@njit(cache=True)
def xy_2_rc(x, y, orig_x, orig_y, orig_c, orig_s, height, width, resolution):
    """
    Translate (x, y) coordinate into (r, c) in the matrix

        Args:
            x (float): coordinate in x (m)
            y (float): coordinate in y (m)
            orig_x (float): x coordinate of the map origin (m)
            orig_y (float): y coordinate of the map origin (m)
        
        Returns:
            r (int): row number in the transform matrix of the given point
            c (int): column number in the transform matrix of the given point
    """
    # translation
    x_trans = x - orig_x
    y_trans = y - orig_y

    # rotation
    x_rot = x_trans * orig_c + y_trans * orig_s
    y_rot = -x_trans * orig_s + y_trans * orig_c

    # clip the state to be a cell
    if x_rot < 0 or x_rot >= width * resolution or y_rot < 0 or y_rot >= height * resolution:
        c = -1
        r = -1
    else:
        c = int(x_rot/resolution)
        r = int(y_rot/resolution)

    return r, c

@njit(cache=True)
def distance_transform(x, y, orig_x, orig_y, orig_c, orig_s, height, width, resolution, dt):
    """
    Look up corresponding distance in the distance matrix

        Args:
            x (float): x coordinate of the lookup point
            y (float): y coordinate of the lookup point
            orig_x (float): x coordinate of the map origin (m)
            orig_y (float): y coordinate of the map origin (m)

        Returns:
            distance (float): corresponding shortest distance to obstacle in meters
    """
    r, c = xy_2_rc(x, y, orig_x, orig_y, orig_c, orig_s, height, width, resolution)
    distance = dt[r, c]
    return distance

@njit(cache=True)
def trace_ray(x, y, theta_index, sines, cosines, eps, orig_x, orig_y, orig_c, orig_s, height, width, resolution, dt, max_range):
    """
    Find the length of a specific ray at a specific scan angle theta
    Purely math calculation and loops, should be JITted.

        Args:
            x (float): current x coordinate of the ego (scan) frame
            y (float): current y coordinate of the ego (scan) frame
            theta_index(int): current index of the scan beam in the scan range
            sines (numpy.ndarray (n, )): pre-calculated sines of the angle array
            cosines (numpy.ndarray (n, )): pre-calculated cosines ...

        Returns:
            total_distance (float): the distance to first obstacle on the current scan beam
    """
    
    # int casting, and index precal trigs
    theta_index_ = int(theta_index)
    s = sines[theta_index_]
    c = cosines[theta_index_]

    # distance to nearest initialization
    dist_to_nearest = distance_transform(x, y, orig_x, orig_y, orig_c, orig_s, height, width, resolution, dt)
    total_dist = dist_to_nearest

    # ray tracing iterations
    while dist_to_nearest > eps and total_dist <= max_range:
        # move in the direction of the ray by dist_to_nearest
        x += dist_to_nearest * c
        y += dist_to_nearest * s

        # update dist_to_nearest for current point on ray
        # also keeps track of total ray length
        dist_to_nearest = distance_transform(x, y, orig_x, orig_y, orig_c, orig_s, height, width, resolution, dt)
        total_dist += dist_to_nearest

    if total_dist > max_range:
        total_dist = max_range
    
    return total_dist

@njit(cache=True)
def get_scan(pose, theta_dis, fov, num_beams, theta_index_increment, sines, cosines, eps, orig_x, orig_y, orig_c, orig_s, height, width, resolution, dt, max_range):
    """
    Perform the scan for each discretized angle of each beam of the laser, loop heavy, should be JITted

        Args:
            pose (numpy.ndarray(3, )): current pose of the scan frame in the map
            theta_dis (int): number of steps to discretize the angles between 0 and 2pi for look up
            fov (float): field of view of the laser scan
            num_beams (int): number of beams in the scan
            theta_index_increment (float): increment between angle indices after discretization

        Returns:
            scan (numpy.ndarray(n, )): resulting laser scan at the pose, n=num_beams
    """
    # empty scan array init
    scan = np.empty((num_beams,))

    # make theta discrete by mapping the range [-pi, pi] onto [0, theta_dis]
    theta_index = theta_dis * (pose[2] - fov/2.)/(2. * np.pi)

    # make sure it's wrapped properly
    theta_index = np.fmod(theta_index, theta_dis)
    while (theta_index < 0):
        theta_index += theta_dis

    # sweep through each beam
    for i in range(0, num_beams):
        # trace the current beam
        scan[i] = trace_ray(pose[0], pose[1], theta_index, sines, cosines, eps, orig_x, orig_y, orig_c, orig_s, height, width, resolution, dt, max_range)

        # increment the beam index
        theta_index += theta_index_increment

        # make sure it stays in the range [0, theta_dis)
        while theta_index >= theta_dis:
            theta_index -= theta_dis

    return scan

@njit(cache=True)
def check_ttc_jit(scan, vel, scan_angles, cosines, side_distances, ttc_thresh):
    """
    Checks the iTTC of each beam in a scan for collision with environment

    Args:
        scan (np.ndarray(num_beams, )): current scan to check
        vel (float): current velocity
        scan_angles (np.ndarray(num_beams, )): precomped angles of each beam
        cosines (np.ndarray(num_beams, )): precomped cosines of the scan angles
        side_distances (np.ndarray(num_beams, )): precomped distances at each beam from the laser to the sides of the car
        ttc_thresh (float): threshold for iTTC for collision

    Returns:
        in_collision (bool): whether vehicle is in collision with environment
        collision_angle (float): at which angle the collision happened
    """
    if vel != 0.0:
        num_beams = scan.shape[0]
        for i in range(num_beams):
            proj_vel = vel*cosines[i]
            ttc = (scan[i] - side_distances[i])/proj_vel
            if (ttc < ttc_thresh) and (ttc >= 0.0):
                in_collision = True
                break
    else:
        in_collision = False

    return in_collision

@njit(cache=True)
def cross(v1, v2):
    """
    Cross product of two 2-vectors

    Args:
        v1, v2 (np.ndarray(2, )): input vectors

    Returns:
        crossproduct (float): cross product
    """
    return v1[0]*v2[1]-v1[1]*v2[0]

@njit(cache=True)
def are_collinear(pt_a, pt_b, pt_c):
    """
    Checks if three points are collinear in 2D

    Args:
        pt_a, pt_b, pt_c (np.ndarray(2, )): points to check in 2D

    Returns:
        col (bool): whether three points are collinear
    """
    tol = 1e-8
    ba = pt_b - pt_a
    ca = pt_a - pt_c
    col = np.fabs(cross(ba, ca)) < tol
    return col

@njit(cache=True)
def get_range(pose, beam_theta, va, vb):
    """
    Get the distance at a beam angle to the vector formed by two of the four vertices of a vehicle

    Args:
        pose (np.ndarray(3, )): pose of the scanning vehicle
        beam_theta (float): angle of the current beam (world frame)
        va, vb (np.ndarray(2, )): the two vertices forming an edge

    Returns:
        distance (float): smallest distance at beam theta from scanning pose to edge
    """
    o = pose[0:2]
    v1 = o - va
    v2 = vb - va
    v3 = np.array([np.cos(beam_theta + np.pi/2.), np.sin(beam_theta + np.pi/2.)])

    denom = v2.dot(v3)
    distance = np.inf

    if np.fabs(denom) > 0.0:
        d1 = cross(v2, v1) / denom
        d2 = v1.dot(v3) / denom
        if d1 >= 0.0 and d2 >= 0.0 and d2 <= 1.0:
            distance = d1
    elif are_collinear(o, va, vb):
        da = np.linalg.norm(va - o)
        db = np.linalg.norm(vb - o)
        distance = min(da, db)

    return distance

@njit(cache=True)
def ray_cast(pose, scan, scan_angles, vertices):
    """
    Modify a scan by ray casting onto another agent's four vertices

    Args:
        pose (np.ndarray(3, )): pose of the vehicle performing scan
        scan (np.ndarray(num_beams, )): original scan to modify
        scan_angles (np.ndarray(num_beams, )): corresponding beam angles
        vertices (np.ndarray(4, 2)): four vertices of a vehicle pose
    
    Returns:
        new_scan (np.ndarray(num_beams, )): modified scan
    """
    num_beams = scan.shape[0]

    # pad vertices so loops around
    looped_vertices = np.empty((5, 2))
    looped_vertices[0:4, :] = vertices
    looped_vertices[4, :] = vertices[0, :]

    # looping over beams
    for i in range(num_beams):
        # looping over vertices
        for j in range(4):
            # check if original scan is longer than ray casted distance
            scan_range = get_range(pose, pose[2]+scan_angles[i], looped_vertices[j,:], looped_vertices[j+1,:])
            if scan_range < scan[i]:
                scan[i] = scan_range

    return scan

class ScanSimulator2D(object):
    """
    2D LIDAR scan simulator class

    Init params:
        num_beams (int): number of beams in the scan
        fov (float): field of view of the laser scan
        eps (float, default=0.0001): ray tracing iteration termination condition
        theta_dis (int, default=2000): number of steps to discretize the angles between 0 and 2pi for look up
        max_range (float, default=30.0): maximum range of the laser
    """

    def __init__(self, num_beams, fov, eps=0.0001, theta_dis=2000, max_range=30.0):
        # initialization 
        self.num_beams = num_beams
        self.fov = fov
        self.eps = eps
        self.theta_dis = theta_dis
        self.max_range = max_range
        self.angle_increment = self.fov / (self.num_beams - 1)
        self.theta_index_increment = theta_dis * self.angle_increment / (2. * np.pi)
        self.orig_c = None
        self.orig_s = None
        self.orig_x = None
        self.orig_y = None
        self.map_height = None
        self.map_width = None
        self.map_resolution = None
        self.dt = None
        self.map_img = None
        
        # precomputing corresponding cosines and sines of the angle array
        theta_arr = np.linspace(0.0, 2*np.pi, num=theta_dis)
        self.sines = np.sin(theta_arr)
        self.cosines = np.cos(theta_arr)
    
    def set_map(self, map_path, map_ext):
        """
        Set the bitmap of the scan simulator by path

            Args:
                map_path (str): path to the map yaml file
                map_ext (str): extension (image type) of the map image

            Returns:
                flag (bool): if image reading and loading is successful
        """
        # TODO: do we open the option to flip the images, and turn rgb into grayscale? or specify the exact requirements in documentation.
        # TODO: throw error if image specification isn't met

        with open(map_path, 'r') as yaml_stream:
            try:
                map_metadata = yaml.safe_load(yaml_stream)
                self.map_resolution = map_metadata['resolution']
                self.origin = map_metadata['origin']
                map_img_name = map_metadata['image']
            except yaml.YAMLError as ex:
                print(ex)

        map_img_path = "maps/"+ map_img_name.split("_")[0]+"/" + map_img_name
        # map_img_path = os.path.splitext(map_path)[0] + map_ext
        self.map_img = np.array(Image.open(map_img_path).transpose(Image.FLIP_TOP_BOTTOM))
        self.map_img = self.map_img.astype(np.float64)

        # grayscale -> binary
        self.map_img[self.map_img <= 128.] = 0.
        self.map_img[self.map_img > 128.] = 255.

        self.map_height = self.map_img.shape[0]
        self.map_width = self.map_img.shape[1]

        # calculate map parameters
        self.orig_x = self.origin[0]
        self.orig_y = self.origin[1]
        self.orig_s = np.sin(self.origin[2])
        self.orig_c = np.cos(self.origin[2])

        # get the distance transform
        self.dt = get_dt(self.map_img, self.map_resolution)

        return True

    def scan(self, pose, rng, std_dev=0.01):
        """
        Perform simulated 2D scan by pose on the given map

            Args:
                pose (numpy.ndarray (3, )): pose of the scan frame (x, y, theta)
                rng (numpy.random.Generator): random number generator to use for whitenoise in scan, or None
                std_dev (float, default=0.01): standard deviation of the generated whitenoise in the scan

            Returns:
                scan (numpy.ndarray (n, )): data array of the laserscan, n=num_beams

            Raises:
                ValueError: when scan is called before a map is set
        """
        
        if self.map_height is None:
            raise ValueError('Map is not set for scan simulator.')
        
        scan = get_scan(pose, self.theta_dis, self.fov, self.num_beams, self.theta_index_increment, self.sines, self.cosines, self.eps, self.orig_x, self.orig_y, self.orig_c, self.orig_s, self.map_height, self.map_width, self.map_resolution, self.dt, self.max_range)

        if rng is not None:
            noise = rng.normal(0., std_dev, size=self.num_beams)
            scan += noise
            
        return scan

    def get_increment(self):
        return self.angle_increment

    def check_location(self, pose):
        if check_bounds(pose[0], pose[1], self.orig_x, self.orig_y, self.orig_c, self.orig_s, self.map_height, self.map_width, self.map_resolution):
            return True
        d = distance_transform(pose[0], pose[1], self.orig_x, self.orig_y, self.orig_c, self.orig_s, self.map_height, self.map_width, self.map_resolution, self.dt)
        if d < 0.001: #1mm
            return True
        return False


@njit(cache=True)
def check_bounds(x, y, orig_x, orig_y, orig_c, orig_s, height, width, resolution):

    # translation
    x_trans = x - orig_x
    y_trans = y - orig_y

    # rotation
    x_rot = x_trans * orig_c + y_trans * orig_s
    y_rot = -x_trans * orig_s + y_trans * orig_c

    # clip the state to be a cell
    if x_rot < 0 or x_rot >= width * resolution or y_rot < 0 or y_rot >= height * resolution:
        return True
    return False

In [ ]:
# opengl stuff

# other

# helpers

# zooming constants
ZOOM_IN_FACTOR = 1.2
ZOOM_OUT_FACTOR = 1/ZOOM_IN_FACTOR

# vehicle shape constants
CAR_LENGTH = 0.58
CAR_WIDTH = 0.31

class EnvRenderer(pyglet.window.Window):
    """
    A window class inherited from pyglet.window.Window, handles the camera/projection interaction, resizing window, and rendering the environment
    """
    def __init__(self, width, height, *args, **kwargs):
        """
        Class constructor

        Args:
            width (int): width of the window
            height (int): height of the window

        Returns:
            None
        """
        conf = Config(sample_buffers=1,
                      samples=4,
                      depth_size=16,
                      double_buffer=True)
        super().__init__(width, height, config=conf, resizable=True, vsync=False, *args, **kwargs)

        # gl init
        glClearColor(9/255, 32/255, 87/255, 1.)

        # initialize camera values
        self.left = -width/2
        self.right = width/2
        self.bottom = -height/2
        self.top = height/2
        self.zoom_level = 1.2
        self.zoomed_width = width
        self.zoomed_height = height

        # current batch that keeps track of all graphics
        self.batch = pyglet.graphics.Batch()

        # current env map
        self.map_points = None
        
        # current env agent poses, (num_agents, 3), columns are (x, y, theta)
        self.poses = None

        # current env agent vertices, (num_agents, 4, 2), 2nd and 3rd dimensions are the 4 corners in 2D
        self.vertices = None

        # obstacles
        self.obstacle_poses = None
        self.obstacles = None

        # current score label
        self.score_label = pyglet.text.Label(
                'Lap Time: {laptime:.2f}, Ego Lap Count: {count:.0f}'.format(
                    laptime=0.0, count=0.0),
                font_size=36,
                x=0,
                y=-800,
                anchor_x='center',
                anchor_y='center',
                # width=0.01,
                # height=0.01,
                color=(255, 255, 255, 255),
                batch=self.batch)

        self.fps_display = pyglet.window.FPSDisplay(self)

    def update_map(self, map_name, map_ext):
        """
        Update the map being drawn by the renderer. Converts image to a list of 3D points representing each obstacle pixel in the map.

        Args:
            map_path (str): absolute path to the map without extensions
            map_ext (str): extension for the map image file

        Returns:
            None
        """

        # load map metadata
        map_path = "maps/" + map_name  
        with open(map_path+ '.yaml', 'r') as yaml_stream:
            try:
                map_metadata = yaml.safe_load(yaml_stream)
                map_resolution = map_metadata['resolution']
                origin = map_metadata['origin']
                origin_x = origin[0]
                origin_y = origin[1]
                map_img_name = map_metadata['image']
            except yaml.YAMLError as ex:
                print(ex)

        map_img_path = "maps/" + map_img_name
        map_img = np.array(Image.open(map_img_path).transpose(Image.FLIP_TOP_BOTTOM)).astype(np.float64)
        map_height = map_img.shape[0]
        map_width = map_img.shape[1]

        # convert map pixels to coordinates
        range_x = np.arange(map_width)
        range_y = np.arange(map_height)
        map_x, map_y = np.meshgrid(range_x, range_y)
        map_x = (map_x * map_resolution + origin_x).flatten()
        map_y = (map_y * map_resolution + origin_y).flatten()
        map_z = np.zeros(map_y.shape)
        map_coords = np.vstack((map_x, map_y, map_z))

        # mask and only leave the obstacle points
        map_mask = map_img == 0.0
        map_mask_flat = map_mask.flatten()
        map_points = 50. * map_coords[:, map_mask_flat].T
        for i in range(map_points.shape[0]):
            self.batch.add(1, GL_POINTS, None, ('v3f/stream', [map_points[i, 0], map_points[i, 1], map_points[i, 2]]), ('c3B/stream', [183, 193, 222]))
        self.map_points = map_points

    def add_obstacles(self, obs_locations, obs_size):
        """
        Adds green obstacles to the map

        Args:
            obs_locations (np.ndarray(n_obstacles, 2)): coordinates of obstacles to add
            obs_size (list(2)): size of obstacles
        
        Returns:
            None
        """
        if self.obstacle_poses is None:
            self.obstacles = []
            for i in range(len(obs_locations)):
                vertices_np = get_vertices(np.array([0., 0., 0.]), obs_size[0], obs_size[1])
                vertices = list(vertices_np.flatten())
                # obstacle = self.batch.add(4, GL_QUADS, None, ('v2f', vertices), ('c3B', [172, 97, 185, 172, 97, 185, 172, 97, 185, 172, 97, 185]))
                obstacle = self.batch.add(4, GL_QUADS, None, ('v2f', vertices), ('c3B', [0, 225, 0, 0, 225, 0, 0, 225, 0, 0, 225, 0]))
                self.obstacles.append(obstacle)

        poses_x = obs_locations[:, 0]
        poses_y = obs_locations[:, 1]
        poses_theta = np.ones_like(poses_x) * np.pi/2

        poses = np.stack((poses_x, poses_y, poses_theta)).T
        for j in range(poses.shape[0]):
            vertices_np = 50. * get_vertices(poses[j, :], obs_size[0], obs_size[1])
            vertices = list(vertices_np.flatten())
            self.obstacles[j].vertices = vertices
        self.obstacle_poses = poses

    def on_resize(self, width, height):
        """
        Callback function on window resize, overrides inherited method, and updates camera values on top of the inherited on_resize() method.

        Potential improvements on current behavior: zoom/pan resets on window resize.

        Args:
            width (int): new width of window
            height (int): new height of window

        Returns:
            None
        """

        # call overrided function
        super().on_resize(width, height)

        # update camera value
        (width, height) = self.get_size()
        self.left = -self.zoom_level * width/2
        self.right = self.zoom_level * width/2
        self.bottom = -self.zoom_level * height/2
        self.top = self.zoom_level * height/2
        self.zoomed_width = self.zoom_level * width
        self.zoomed_height = self.zoom_level * height

    def on_mouse_drag(self, x, y, dx, dy, buttons, modifiers):
        """
        Callback function on mouse drag, overrides inherited method.

        Args:
            x (int): Distance in pixels from the left edge of the window.
            y (int): Distance in pixels from the bottom edge of the window.
            dx (int): Relative X position from the previous mouse position.
            dy (int): Relative Y position from the previous mouse position.
            buttons (int): Bitwise combination of the mouse buttons currently pressed.
            modifiers (int): Bitwise combination of any keyboard modifiers currently active.

        Returns:
            None
        """

        # pan camera
        self.left -= dx * self.zoom_level
        self.right -= dx * self.zoom_level
        self.bottom -= dy * self.zoom_level
        self.top -= dy * self.zoom_level

    def on_mouse_scroll(self, x, y, dx, dy):
        """
        Callback function on mouse scroll, overrides inherited method.

        Args:
            x (int): Distance in pixels from the left edge of the window.
            y (int): Distance in pixels from the bottom edge of the window.
            scroll_x (float): Amount of movement on the horizontal axis.
            scroll_y (float): Amount of movement on the vertical axis.

        Returns:
            None
        """

        # Get scale factor
        f = ZOOM_IN_FACTOR if dy > 0 else ZOOM_OUT_FACTOR if dy < 0 else 1

        # If zoom_level is in the proper range
        if .01 < self.zoom_level * f < 10:

            self.zoom_level *= f

            (width, height) = self.get_size()

            mouse_x = x/width
            mouse_y = y/height

            mouse_x_in_world = self.left + mouse_x*self.zoomed_width
            mouse_y_in_world = self.bottom + mouse_y*self.zoomed_height

            self.zoomed_width *= f
            self.zoomed_height *= f

            self.left = mouse_x_in_world - mouse_x * self.zoomed_width
            self.right = mouse_x_in_world + (1 - mouse_x) * self.zoomed_width
            self.bottom = mouse_y_in_world - mouse_y * self.zoomed_height
            self.top = mouse_y_in_world + (1 - mouse_y) * self.zoomed_height

    def on_close(self):
        """
        Callback function when the 'x' is clicked on the window, overrides inherited method. Also throws exception to end the python program when in a loop.

        Args:
            None

        Returns:
            None

        Raises:
            Exception: with a message that indicates the rendering window was closed
        """

        super().on_close()
        raise Exception('Rendering window was closed.')

    def on_draw(self):
        """
        Function when the pyglet is drawing. The function draws the batch created that includes the map points, the agent polygons, and the information text, and the fps display.
        
        Args:
            None

        Returns:
            None
        """

        # if map and poses doesn't exist, raise exception
        if self.map_points is None:
            raise Exception('Map not set for renderer.')
        if self.poses is None:
            raise Exception('Agent poses not updated for renderer.')

        # Initialize Projection matrix
        glMatrixMode(GL_PROJECTION)
        glLoadIdentity()

        # Initialize Modelview matrix
        glMatrixMode(GL_MODELVIEW)
        glLoadIdentity()
        # Save the default modelview matrix
        glPushMatrix()

        # Clear window with ClearColor
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        # Set orthographic projection matrix
        glOrtho(self.left, self.right, self.bottom, self.top, 1, -1)

        # Draw all batches
        self.batch.draw()
        self.fps_display.draw()
        # Remove default modelview matrix
        glPopMatrix()

    def update_obs(self, obs):
        """
        Updates the renderer with the latest observation from the gym environment, including the agent poses, and the information text.

        Args:
            obs (dict): observation dict from the gym env

        Returns:
            None
        """

        self.ego_idx = obs['ego_idx']
        poses_x = obs['poses_x']
        poses_y = obs['poses_y']
        poses_theta = obs['poses_theta']

        num_agents = len(poses_x)
        if self.poses is None:
            self.cars = []
            for i in range(num_agents):
                if i == self.ego_idx:
                    vertices_np = get_vertices(np.array([0., 0., 0.]), CAR_LENGTH, CAR_WIDTH)
                    vertices = list(vertices_np.flatten())
                    car = self.batch.add(4, GL_QUADS, None, ('v2f', vertices), ('c3B', [172, 97, 185, 172, 97, 185, 172, 97, 185, 172, 97, 185]))
                    self.cars.append(car)
                else:
                    vertices_np = get_vertices(np.array([0., 0., 0.]), CAR_LENGTH, CAR_WIDTH)
                    vertices = list(vertices_np.flatten())
                    car = self.batch.add(4, GL_QUADS, None, ('v2f', vertices), ('c3B', [99, 52, 94, 99, 52, 94, 99, 52, 94, 99, 52, 94]))
                    self.cars.append(car)

        poses = np.stack((poses_x, poses_y, poses_theta)).T
        for j in range(poses.shape[0]):
            vertices_np = 50. * get_vertices(poses[j, :], CAR_LENGTH, CAR_WIDTH)
            vertices = list(vertices_np.flatten())
            self.cars[j].vertices = vertices
        self.poses = poses

        self.score_label.text = 'Lap Time: {laptime:.2f}, Ego Lap Count: {count:.0f}'.format(laptime=obs['lap_times'][0], count=obs['lap_counts'][obs['ego_idx']])

In [ ]:
class RaceCarHistory(object):
    def __init__(self):
        self.states = []
        self.steer_acts = []
        self.vel_acts = []

    def plot_history(self):
        states = np.array(self.states)
        # plt.figure(1)
        # plt.clf()
        # plt.title('RaceCar Locations')
        # plt.plot(states[:,0], states[:,1])

        plt.figure(2)
        plt.clf()
        plt.title('RaceCar Steering')
        plt.plot(self.steer_acts)
        plt.plot(states[:, 2])

        plt.figure(3)
        plt.clf()
        plt.title('RaceCar Velocity')
        plt.plot(self.vel_acts)
        plt.plot(states[:, 3])

        # plt.figure(4)
        # plt.clf()
        # plt.title('RaceCar Slip angle')
        # plt.plot(states[:, 6])

        # plt.figure(5)
        # plt.clf()
        # plt.title('RaceCar Yaw rate')
        # plt.plot(states[:, 5])

        # plt.figure(6)
        # plt.clf()
        # plt.title('RaceCar Vy')
        # vy = np.tan(states[:, 6]) * states[:,3]
        # plt.plot(vy)

        plt.show()
        self.reset_history()

    def reset_history(self):
        self.states = []
        self.steer_acts = []
        self.vel_acts = []


class RaceCar(object):
    """
    Base level race car class, handles the physics and laser scan of a single vehicle

    Data Members:
        params (dict): vehicle parameters dictionary
        is_ego (bool): ego identifier
        time_step (float): physics timestep
        num_beams (int): number of beams in laser
        fov (float): field of view of laser
        state (np.ndarray (7, )): state vector [x, y, theta, vel, steer_angle, ang_vel, slip_angle]
        odom (np.ndarray(13, )): odometry vector [x, y, z, qx, qy, qz, qw, linear_x, linear_y, linear_z, angular_x, angular_y, angular_z]
        accel (float): current acceleration input
        steer_angle_vel (float): current steering velocity input
        in_collision (bool): collision indicator

    """

    # static objects that don't need to be stored in class instances
    scan_simulator = None
    cosines = None
    scan_angles = None
    side_distances = None

    def __init__(self, params, seed, is_ego=False, time_step=0.01, num_beams=1080, fov=4.7):
        """
        Init function

        Args:
            params (dict): vehicle parameter dictionary, includes {'mu', 'C_Sf', 'C_Sr', 'lf', 'lr', 'h', 'm', 'I', 's_min', 's_max', 'sv_min', 'sv_max', 'v_switch', 'a_max': 9.51, 'v_min', 'v_max', 'length', 'width'}
            is_ego (bool, default=False): ego identifier
            time_step (float, default=0.01): physics sim time step
            num_beams (int, default=1080): number of beams in the laser scan
            fov (float, default=4.7): field of view of the laser

        Returns:
            None
        """

        # initialization
        self.params = params
        self.seed = seed
        self.is_ego = is_ego
        self.time_step = time_step
        self.num_beams = num_beams
        self.fov = fov
        self.history = RaceCarHistory()
        # state is [x, y, steer_angle, vel, yaw_angle, yaw_rate, slip_angle]
        self.state = np.zeros((7, ))

        # pose of opponents in the world
        self.opp_poses = None

        # control inputs
        self.accel = 0.0
        self.steer_angle_vel = 0.0

        # steering delay buffer
        self.steer_buffer = np.empty((0, ))
        self.steer_buffer_size = 2

        # collision identifier
        self.in_collision = False

        # collision threshold for iTTC to environment
        self.ttc_thresh = 0.005

        # initialize scan sim
        self.scan_rng = np.random.default_rng(seed=self.seed)
        if RaceCar.scan_simulator is None:
            RaceCar.scan_simulator = ScanSimulator2D(num_beams, fov)

            scan_ang_incr = RaceCar.scan_simulator.get_increment()

            # angles of each scan beam, distance from lidar to edge of car at each beam, and precomputed cosines of each angle
            RaceCar.cosines = np.zeros((num_beams, ))
            RaceCar.scan_angles = np.zeros((num_beams, ))
            RaceCar.side_distances = np.zeros((num_beams, ))

            dist_sides = params['width']/2.
            dist_fr = (params['lf']+params['lr'])/2.

            for i in range(num_beams):
                angle = -fov/2. + i*scan_ang_incr
                RaceCar.scan_angles[i] = angle
                RaceCar.cosines[i] = np.cos(angle)

                if angle > 0:
                    if angle < np.pi/2:
                        # between 0 and pi/2
                        to_side = dist_sides / np.sin(angle)
                        to_fr = dist_fr / np.cos(angle)
                        RaceCar.side_distances[i] = min(to_side, to_fr)
                    else:
                        # between pi/2 and pi
                        to_side = dist_sides / np.cos(angle - np.pi/2.)
                        to_fr = dist_fr / np.sin(angle - np.pi/2.)
                        RaceCar.side_distances[i] = min(to_side, to_fr)
                else:
                    if angle > -np.pi/2:
                        # between 0 and -pi/2
                        to_side = dist_sides / np.sin(-angle)
                        to_fr = dist_fr / np.cos(-angle)
                        RaceCar.side_distances[i] = min(to_side, to_fr)
                    else:
                        # between -pi/2 and -pi
                        to_side = dist_sides / np.cos(-angle - np.pi/2)
                        to_fr = dist_fr / np.sin(-angle - np.pi/2)
                        RaceCar.side_distances[i] = min(to_side, to_fr)

    def update_params(self, params):
        """
        Updates the physical parameters of the vehicle
        Note that does not need to be called at initialization of class anymore

        Args:
            params (dict): new parameters for the vehicle

        Returns:
            None
        """
        self.params = params
    
    def set_map(self, map_path, map_ext):
        """
        Sets the map for scan simulator
        
        Args:
            map_path (str): absolute path to the map yaml file
            map_ext (str): extension of the map image file
        """
        RaceCar.scan_simulator.set_map(map_path, map_ext)

    def reset(self, pose):
        """
        Resets the vehicle to a pose
        
        Args:
            pose (np.ndarray (3, )): pose to reset the vehicle to

        Returns:
            None
        """
        # clear control inputs
        self.accel = 0.0
        self.steer_angle_vel = 0.0
        # clear collision indicator
        self.in_collision = False
        # clear state
        self.state = np.zeros((7, ))
        self.state[0:2] = pose[0:2]
        self.state[4] = pose[2]
        self.steer_buffer = np.empty((0, ))
        # reset scan random generator
        # self.scan_rng = np.random.default_rng(seed=self.seed)
        self.history.reset_history()

    def ray_cast_agents(self, scan):
        """
        Ray cast onto other agents in the env, modify original scan

        Args:
            scan (np.ndarray, (n, )): original scan range array

        Returns:
            new_scan (np.ndarray, (n, )): modified scan
        """

        # starting from original scan
        new_scan = scan
        
        # loop over all opponent vehicle poses
        for opp_pose in self.opp_poses:
            # get vertices of current oppoenent
            opp_vertices = get_vertices(opp_pose, self.params['length'], self.params['width'])

            new_scan = ray_cast(np.append(self.state[0:2], self.state[4]), new_scan, self.scan_angles, opp_vertices)

        return new_scan

    def check_ttc(self, current_scan):
        """
        Check iTTC against the environment, sets vehicle states accordingly if collision occurs.
        Note that this does NOT check collision with other agents.

        state is [x, y, steer_angle, vel, yaw_angle, yaw_rate, slip_angle]

        Args:
            current_scan

        Returns:
            None
        """
        
        in_collision = check_ttc_jit(current_scan, self.state[3], self.scan_angles, self.cosines, self.side_distances, self.ttc_thresh)

        # if in collision stop vehicle
        if in_collision:
            self.state[3:] = 0.
            self.accel = 0.0
            self.steer_angle_vel = 0.0

        # update state
        self.in_collision = in_collision

        return in_collision

    def update_pose(self, raw_steer, vel):
        """
        Steps the vehicle's physical simulation

        Args:
            steer (float): desired steering angle
            vel (float): desired longitudinal velocity

        Returns:
            current_scan
        """

        # state is [x, y, steer_angle, vel, yaw_angle, yaw_rate, slip_angle]

        # steering delay
        self.history.steer_acts.append(raw_steer)
        self.history.vel_acts.append(vel)
        steer = 0.
        if self.steer_buffer.shape[0] < self.steer_buffer_size:
            steer = 0.
            self.steer_buffer = np.append(raw_steer, self.steer_buffer)
        else:
            steer = self.steer_buffer[-1]
            self.steer_buffer = self.steer_buffer[:-1]
            self.steer_buffer = np.append(raw_steer, self.steer_buffer)


        # steering angle velocity input to steering velocity acceleration input
        accl, sv = pid(vel, steer, self.state[3], self.state[2], self.params['sv_max'], self.params['a_max'], self.params['v_max'], self.params['v_min'])
        
        # update physics, get RHS of diff'eq
        f = vehicle_dynamics_st(
            self.state,
            np.array([sv, accl]),
            self.params['mu'],
            self.params['C_Sf'],
            self.params['C_Sr'],
            self.params['lf'],
            self.params['lr'],
            self.params['h'],
            self.params['m'],
            self.params['I'],
            self.params['s_min'],
            self.params['s_max'],
            self.params['sv_min'],
            self.params['sv_max'],
            self.params['v_switch'],
            self.params['a_max'],
            self.params['v_min'],
            self.params['v_max'])

        # update state
        self.state = self.state + f * self.time_step

        # bound yaw angle
        if self.state[4] > 2*np.pi:
            self.state[4] = self.state[4] - 2*np.pi
        elif self.state[4] < 0:
            self.state[4] = self.state[4] + 2*np.pi

        self.history.states.append(self.state.copy())

        # update scan
        current_scan = RaceCar.scan_simulator.scan(np.append(self.state[0:2], self.state[4]), self.scan_rng)

        return current_scan

    def update_opp_poses(self, opp_poses):
        """
        Updates the vehicle's information on other vehicles

        Args:
            opp_poses (np.ndarray(num_other_agents, 3)): updated poses of other agents

        Returns:
            None
        """
        self.opp_poses = opp_poses


    def update_scan(self, agent_scans, agent_index):
        """
        Steps the vehicle's laser scan simulation
        Separated from update_pose because needs to update scan based on NEW poses of agents in the environment

        Args:
            agent scans list (modified in-place),
            agent index (int)

        Returns:
            None
        """

        current_scan = agent_scans[agent_index]

        # check ttc
        self.check_ttc(current_scan)

        # ray cast other agents to modify scan
        new_scan = self.ray_cast_agents(current_scan)

        agent_scans[agent_index] = new_scan

class Simulator(object):
    """
    Simulator class, handles the interaction and update of all vehicles in the environment

    Data Members:
        num_agents (int): number of agents in the environment
        time_step (float): physics time step
        agent_poses (np.ndarray(num_agents, 3)): all poses of all agents
        agents (list[RaceCar]): container for RaceCar objects
        collisions (np.ndarray(num_agents, )): array of collision indicator for each agent
        collision_idx (np.ndarray(num_agents, )): which agent is each agent in collision with

    """

    def __init__(self, params, num_agents, seed, time_step=0.01, ego_idx=0):
        """
        Init function

        Args:
            params (dict): vehicle parameter dictionary, includes {'mu', 'C_Sf', 'C_Sr', 'lf', 'lr', 'h', 'm', 'I', 's_min', 's_max', 'sv_min', 'sv_max', 'v_switch', 'a_max', 'v_min', 'v_max', 'length', 'width'}
            num_agents (int): number of agents in the environment
            seed (int): seed of the rng in scan simulation
            time_step (float, default=0.01): physics time step
            ego_idx (int, default=0): ego vehicle's index in list of agents

        Returns:
            None
        """
        self.num_agents = num_agents
        self.seed = seed
        self.time_step = time_step
        self.ego_idx = ego_idx
        self.params = params
        self.agent_poses = np.empty((self.num_agents, 3))
        self.agents = []
        self.collisions = np.zeros((self.num_agents, ))
        self.collision_idx = -1 * np.ones((self.num_agents, ))

        # obs info
        self.obs_locations = None
        self.obs_size = None

        # initializing agents
        fov = np.pi 
        n_beams = 20
        #TODO: add in a parameter file for each car that can be passed on
        for i in range(self.num_agents):
            if i == ego_idx:
                ego_car = RaceCar(params, self.seed, is_ego=True, num_beams=n_beams, fov=fov)
                self.agents.append(ego_car)
            else:
                agent = RaceCar(params, self.seed)
                self.agents.append(agent)

    def set_map(self, map_path, map_ext):
        """
        Sets the map of the environment and sets the map for scan simulator of each agent

        Args:
            map_path (str): path to the map yaml file
            map_ext (str): extension for the map image file

        Returns:
            None
        """
        for agent in self.agents:
            agent.set_map(map_path, map_ext)

    def update_map_img(self, map_img):
        """
        Updates the map image used in the scan simulator.
        This function is intended for used when adding obstacles to the map.

        Args:
            map_img (np.ndarray(map_width, map_height)): new image to be used

        Returns:
            None
        """
        for agent in self.agents:
            agent.scan_simulator.update_map_img(map_img)

    def update_params(self, params, agent_idx=-1):
        """
        Updates the params of agents, if an index of an agent is given, update only that agent's params

        Args:
            params (dict): dictionary of params, see details in docstring of __init__
            agent_idx (int, default=-1): index for agent that needs param update, if negative, update all agents

        Returns:
            None
        """
        if agent_idx < 0:
            # update params for all
            for agent in self.agents:
                agent.update_params(params)
        elif agent_idx >= 0 and agent_idx < self.num_agents:
            # only update one agent's params
            self.agents[agent_idx].update_params(params)
        else:
            # index out of bounds, throw error
            raise IndexError('Index given is out of bounds for list of agents.')

    def check_collision(self):
        """
        Checks for collision between agents using GJK and agents' body vertices

        Args:
            None

        Returns:
            None
        """
        # get vertices of all agents
        all_vertices = np.empty((self.num_agents, 4, 2))
        for i in range(self.num_agents):
            all_vertices[i, :, :] = get_vertices(np.append(self.agents[i].state[0:2],self.agents[i].state[4]), self.params['length'], self.params['width'])
        self.collisions, self.collision_idx = collision_multiple(all_vertices)


    def step(self, control_inputs):
        """
        Steps the simulation environment

        Args:
            control_inputs (np.ndarray (num_agents, 2)): control inputs of all agents, first column is desired steering angle, second column is desired velocity
        
        Returns:
            observations (dict): dictionary for observations: poses of agents, current laser scan of each agent, collision indicators, etc.
        """


        agent_scans = []

        # looping over agents
        for i, agent in enumerate(self.agents):
            # update each agent's pose
            current_scan = agent.update_pose(control_inputs[i, 0], control_inputs[i, 1])
            agent_scans.append(current_scan)

            # update sim's information of agent poses
            self.agent_poses[i, :] = np.append(agent.state[0:2], agent.state[4])

        # check collisions between all agents
        self.check_collision()

        #! this is bad!!!!!!!!!!!!!!!!
        # removed to prevent instant crashing due to bug.....
        # for i, agent in enumerate(self.agents):
        #     # update agent's information on other agents
        #     opp_poses = np.concatenate((self.agent_poses[0:i, :], self.agent_poses[i+1:, :]), axis=0)
        #     agent.update_opp_poses(opp_poses)

        #     # update each agent's current scan based on other agents
        #     agent.update_scan(agent_scans, i)

        #     # update agent collision with environment
        #     if agent.in_collision:
        #         self.collisions[i] = 1.

        # fill in observations
        # state is [x, y, steer_angle, vel, yaw_angle, yaw_rate, slip_angle]
        # collision_angles is removed from observations
        observations = {'ego_idx': self.ego_idx,
            'scans': [],
            'poses_x': [],
            'poses_y': [],
            'poses_theta': [],
            'linear_vels_x': [],
            'linear_vels_y': [],
            'ang_vels_z': [],
            'steering_deltas': [],
            'full_states': [],
            'collisions': self.collisions}
        for i, agent in enumerate(self.agents):
            observations['scans'].append(agent_scans[i])
            observations['poses_x'].append(agent.state[0])
            observations['poses_y'].append(agent.state[1])
            observations['poses_theta'].append(agent.state[4])
            observations['linear_vels_x'].append(agent.state[3])
            observations['linear_vels_y'].append(0.)
            observations['steering_deltas'].append(agent.state[2])
            observations['ang_vels_z'].append(agent.state[5])
            observations['full_states'].append(agent.state)

        return observations

    def reset(self, poses):
        """
        Resets the simulation environment by given poses

        Arges:
            poses (np.ndarray (num_agents, 3)): poses to reset agents to

        Returns:
            None
        """
        
        if poses.shape[0] != self.num_agents:
            raise ValueError('Number of poses for reset does not match number of agents.')

        # loop over poses to reset
        for i in range(self.num_agents):
            self.agents[i].reset(poses[i, :])

In [ ]:
# gym imports

# base classes

# others

# gl
# constants

# rendering
VIDEO_W = 600
VIDEO_H = 400
WINDOW_W = 1000
WINDOW_H = 800



class F110Env(gym.Env):
    """
    OpenAI gym environment for F1TENTH
    
    Env should be initialized by calling gym.make('f110_gym:f110-v0', **kwargs)

    Args:
        kwargs:
            seed (int, default=12345): seed for random state and reproducibility
            
            map (str, default='vegas'): name of the map used for the environment. Currently, available environments include: 'berlin', 'vegas', 'skirk'. You could use a string of the absolute path to the yaml file of your custom map.
        
            map_ext (str, default='png'): image extension of the map image file. For example 'png', 'pgm'
        
            params (dict, default={'mu': 1.0489, 'C_Sf':, 'C_Sr':, 'lf': 0.15875, 'lr': 0.17145, 'h': 0.074, 'm': 3.74, 'I': 0.04712, 's_min': -0.4189, 's_max': 0.4189, 'sv_min': -3.2, 'sv_max': 3.2, 'v_switch':7.319, 'a_max': 9.51, 'v_min':-5.0, 'v_max': 20.0, 'width': 0.31, 'length': 0.58}): dictionary of vehicle parameters.
            mu: surface friction coefficient
            C_Sf: Cornering stiffness coefficient, front
            C_Sr: Cornering stiffness coefficient, rear
            lf: Distance from center of gravity to front axle
            lr: Distance from center of gravity to rear axle
            h: Height of center of gravity
            m: Total mass of the vehicle
            I: Moment of inertial of the entire vehicle about the z axis
            s_min: Minimum steering angle constraint
            s_max: Maximum steering angle constraint
            sv_min: Minimum steering velocity constraint
            sv_max: Maximum steering velocity constraint
            v_switch: Switching velocity (velocity at which the acceleration is no longer able to create wheel spin)
            a_max: Maximum longitudinal acceleration
            v_min: Minimum longitudinal velocity
            v_max: Maximum longitudinal velocity
            width: width of the vehicle in meters
            length: length of the vehicle in meters

            num_agents (int, default=2): number of agents in the environment

            timestep (float, default=0.01): physics timestep

            ego_idx (int, default=0): ego's index in list of agents
    """
    metadata = {'render.modes': ['human', 'human_fast']}

    # rendering
    renderer = None
    current_obs = None
    render_callbacks = []

    def __init__(self, **kwargs):        
        # kwargs extraction
        # print("Env has been made")
        try:
            self.seed = kwargs['seed']
        except:
            self.seed = 12345
        try:
            self.map_name = kwargs['map']
            self.map_path = "maps/" +self.map_name + "/"+ self.map_name + "_map" + ".yaml"
            # different default maps
        #     if self.map_name == 'berlin':
        #         self.map_path = os.path.dirname(os.path.abspath(__file__)) + '/maps/berlin.yaml'
        #     elif self.map_name == 'skirk':
        #         self.map_path = os.path.dirname(os.path.abspath(__file__)) + '/maps/skirk.yaml'
        #     elif self.map_name == 'levine':
        #         self.map_path = os.path.dirname(os.path.abspath(__file__)) + '/maps/levine.yaml'
        #     else:
        #         self.map_path = self.map_name + '.yaml'
        except:
            raise RuntimeError("No map given")
        #     self.map_path = os.path.dirname(os.path.abspath(__file__)) + '/maps/vegas.yaml'

        try:
            self.map_ext = kwargs['map_ext']
        except:
            self.map_ext = '.png'

        try:
            self.params = kwargs['params']
        except:
            self.params = {'mu': 1.0489, 'C_Sf': 4.718, 'C_Sr': 5.4562, 'lf': 0.15875, 'lr': 0.17145, 'h': 0.074, 'm': 3.74, 'I': 0.04712, 's_min': -0.4189, 's_max': 0.4189, 'sv_min': -3.2, 'sv_max': 3.2, 'v_switch': 7.319, 'a_max': 9.51, 'v_min':-5.0, 'v_max': 20.0, 'width': 0.31, 'length': 0.58}

        # simulation parameters
        try:
            self.num_agents = kwargs['num_agents']
        except:
            self.num_agents = 1

        try:
            self.timestep = kwargs['timestep']
        except:
            self.timestep = 0.01

        # default ego index
        try:
            self.ego_idx = kwargs['ego_idx']
        except:
            self.ego_idx = 0

        # radius to consider done
        self.start_thresh = 0.5  # 10cm

        # env states
        self.poses_x = []
        self.poses_y = []
        self.poses_theta = []
        self.collisions = np.zeros((self.num_agents, ))
        # TODO: collision_idx not used yet
        # self.collision_idx = -1 * np.ones((self.num_agents, ))

        # loop completion
        self.near_start = True
        self.num_toggles = 0

        # race info
        self.lap_times = np.zeros((self.num_agents, ))
        self.lap_counts = np.zeros((self.num_agents, ))
        self.current_time = 0.0

        # finish line info
        self.num_toggles = 0
        self.near_start = True
        self.near_starts = np.array([True]*self.num_agents)
        self.toggle_list = np.zeros((self.num_agents,))
        self.start_xs = np.zeros((self.num_agents, ))
        self.start_ys = np.zeros((self.num_agents, ))
        self.start_thetas = np.zeros((self.num_agents, ))
        self.start_rot = np.eye(2)

        # initiate stuff
        self.sim = Simulator(self.params, self.num_agents, self.seed)
        self.sim.set_map(self.map_path, self.map_ext)
        self.empty_map_img = np.copy(self.sim.agents[0].scan_simulator.map_img)

        # stateful observations for rendering
        self.render_obs = None

        self.poses = []

    def __del__(self):
        """
        Finalizer, does cleanup
        """
        pass

    def _check_done(self):
        """
        Check if the current rollout is done
        
        Args:
            None

        Returns:
            done (bool): whether the rollout is done
            toggle_list (list[int]): each agent's toggle list for crossing the finish zone
        """

        # this is assuming 2 agents
        # TODO: switch to maybe s-based
        left_t = 2
        right_t = 2
        
        poses_x = np.array(self.poses_x)-self.start_xs
        poses_y = np.array(self.poses_y)-self.start_ys
        delta_pt = np.dot(self.start_rot, np.stack((poses_x, poses_y), axis=0))
        temp_y = delta_pt[1,:]
        idx1 = temp_y > left_t
        idx2 = temp_y < -right_t
        temp_y[idx1] -= left_t
        temp_y[idx2] = -right_t - temp_y[idx2]
        temp_y[np.invert(np.logical_or(idx1, idx2))] = 0

        dist2 = delta_pt[0, :]**2 + temp_y**2
        # closes = dist2 <= 0.02
        closes = np.sqrt(dist2) <= 0.1
        for i in range(self.num_agents):
            if closes[i] and not self.near_starts[i] and self.current_time > 10:
                self.near_starts[i] = True
                self.toggle_list[i] += 1
            elif not closes[i] and self.near_starts[i]:
                self.near_starts[i] = False
                self.toggle_list[i] += 1
            self.lap_counts[i] = self.toggle_list[i] // 2
            if self.toggle_list[i] < 4:
                self.lap_times[i] = self.current_time
        
        done = (self.collisions[self.ego_idx]) or (np.all(self.toggle_list >= 2) and self.current_time > 10)
        # This number (2) is 2x the number of laps desired
        
        # done = done and self.current_time > 10 #! this is a temporary hack for the porto map
        if self.current_time < 10:
            self.lap_counts[0] = 0

        # if done:
        #     print(f"d2: {dist2[0]} -> Distance: {np.sqrt(dist2[self.ego_idx])}")
        #     print(f"Collisions: {self.collisions[0]}")

        return done, self.toggle_list >= 4

    def check_location(self):
        location = np.array([self.poses_x[0], self.poses_y[0]])
        p_done = self.sim.agents[0].scan_simulator.check_location(location)
        if not p_done:
            return False
        print(f"Personl done called: {location}")
        return True

    def _update_state(self, obs_dict):
        """
        Update the env's states according to observations
        
        Args:
            obs_dict (dict): dictionary of observation

        Returns:
            None
        """
        self.poses_x = obs_dict['poses_x']
        self.poses_y = obs_dict['poses_y']
        self.poses_theta = obs_dict['poses_theta']
        self.collisions = obs_dict['collisions']

    def step(self, action):
        """
        Step function for the gym env

        Args:
            action (np.ndarray(num_agents, 2))

        Returns:
            obs (dict): observation of the current step
            reward (float, default=self.timestep): step reward, currently is physics timestep
            done (bool): if the simulation is done
            info (dict): auxillary information dictionary
        """
        
        # call simulation step
        obs = self.sim.step(action)
        obs['lap_times'] = self.lap_times
        obs['lap_counts'] = self.lap_counts

        F110Env.current_obs = obs

        self.render_obs = {
            'ego_idx': obs['ego_idx'],
            'poses_x': obs['poses_x'],
            'poses_y': obs['poses_y'],
            'poses_theta': obs['poses_theta'],
            'lap_times': obs['lap_times'],
            'lap_counts': obs['lap_counts']
            }
        self.poses.append([obs['poses_x'][0], obs['poses_y'][0]])
        # times
        reward = self.timestep
        self.current_time = self.current_time + self.timestep
        
        # update data member
        self._update_state(obs)

        # check done
        done, toggle_list = self._check_done()
        if self.check_location():
            obs['collisions'][0] = True
            done = True

        info = {'checkpoint_done': toggle_list}

        return obs, reward, done, info

    def reset(self, poses):
        """
        Reset the gym environment by given poses

        Args:
            poses (np.ndarray (num_agents, 3)): poses to reset agents to

        Returns:
            obs (dict): observation of the current step
            reward (float, default=self.timestep): step reward, currently is physics timestep
            done (bool): if the simulation is done
            info (dict): auxillary information dictionary
        """
        # reset counters and data members
        self.current_time = 0.0
        self.collisions = np.zeros((self.num_agents, ))
        self.num_toggles = 0
        self.near_start = True
        self.near_starts = np.array([True]*self.num_agents)
        self.toggle_list = np.zeros((self.num_agents,))

        # states after reset
        self.start_xs = poses[:, 0]
        self.start_ys = poses[:, 1]
        self.start_thetas = poses[:, 2]
        self.start_rot = np.array([[np.cos(-self.start_thetas[self.ego_idx]), -np.sin(-self.start_thetas[self.ego_idx])], [np.sin(-self.start_thetas[self.ego_idx]), np.cos(-self.start_thetas[self.ego_idx])]])

        # call reset to simulator
        self.sim.reset(poses)

        # get no input observations
        action = np.zeros((self.num_agents, 2))
        obs, reward, done, info = self.step(action)

        self.render_obs = {
            'ego_idx': obs['ego_idx'],
            'poses_x': obs['poses_x'],
            'poses_y': obs['poses_y'],
            'poses_theta': obs['poses_theta'],
            'lap_times': obs['lap_times'],
            'lap_counts': obs['lap_counts']
            }

        self.poses = []
        self.poses.append([obs['poses_x'][0], obs['poses_y'][0]])
        
        return obs, reward, done, info

    def data_reset(self):
        """
        Reset the data in the environment so the lap counting works
        """
        self.current_time = 0.0
        self.collisions = np.zeros((self.num_agents, ))
        self.num_toggles = 0
        self.near_start = True
        self.near_starts = np.array([True]*self.num_agents)
        self.toggle_list = np.zeros((self.num_agents,))

    def load_centerline(self, file_name=None):
        """
        Loads a centerline from a csv file. 
        Note: the file must be in the same folder as the map which the simulator loads.

        Args:
            file_name (string): the name of a csv file with the centerline of the track in the form [x_i, y_i, w_l_i, w_r_i], location and width

        Returns:
            center_pts (np.ndarray): the loaded center point location
            widths (np.ndarray): the widths of the track
        """
        if file_name is None:
            map_path = os.path.splitext(self.map_path)[0]
            file_name = map_path + '_centerline.csv'
        else:
            file_name = self.map_path + file_name

        track = []
        with open(file_name, 'r') as csvfile:
            csvFile = csv.reader(csvfile, quoting=csv.QUOTE_NONNUMERIC)  
            for lines in csvFile:  
                track.append(lines)
        track = np.array(track)

        center_pts = track[:, 0:2]
        widths = track[:, 2:4]

        return center_pts, widths

    def add_obstacles(self, n_obstacles, obstacle_size=[0.5, 0.5]):
        """
        Adds a set number of obstacles to the envioronment using the track centerline. 
        Note: this function requires a csv file with the centerline points in it which can be loaded. 
        Updates the renderer and the map kept by the laser scaner for each vehicle in the simulator

        Args:
            n_obstacles (int): number of obstacles to add
            obstacle_size (list(2)): rectangular size of obstacles
            
        Returns:
            None
        """
        map_img = np.copy(self.empty_map_img)
        scan_sim = self.sim.agents[0].scan_simulator

        obs_size_m = np.array(obstacle_size)
        obs_size_px = np.array(obs_size_m / scan_sim.map_resolution, dtype=int)
        
        center_pts, widths = self.load_centerline()

        # randomly select certain idx's
        rand_idxs = np.random.randint(1, len(center_pts)-1, n_obstacles)
        
        # randomly select location within box of minimum_width around the center point
        rands = np.random.uniform(-1, 1, size=(n_obstacles, 2))
        obs_locations = center_pts[rand_idxs, :] + rands * widths[rand_idxs]

        # change the values of the img at each obstacle location
        obs_locations = np.array(obs_locations)
        for location in obs_locations:
            # convert the location to the pixel coordinates
            x = int((location[0] - scan_sim.orig_x) / scan_sim.map_resolution)
            y = int((location[1] - scan_sim.orig_y) / scan_sim.map_resolution)
            map_img[y:y+obs_size_px[0], x:x+obs_size_px[1]] = 0

        # update the image in the simulator
        self.sim.update_map_img(map_img)

        # if rendering is on, then add obstacles to the renderer
        if self.renderer is not None:
            self.renderer.add_obstacles(obs_locations, obs_size_m)

    def update_map(self, map_path, map_ext):
        """
        Updates the map used by simulation

        Args:
            map_path (str): absolute path to the map yaml file
            map_ext (str): extension of the map image file

        Returns:
            None
        """
        self.sim.set_map(map_path, map_ext)

    def update_params(self, params, index=-1):
        """
        Updates the parameters used by simulation for vehicles
        
        Args:
            params (dict): dictionary of parameters
            index (int, default=-1): if >= 0 then only update a specific agent's params

        Returns:
            None
        """
        self.sim.update_params(params, agent_idx=index)

    def add_render_callback(self, callback_func):
        """
        Add extra drawing function to call during rendering.

        Args:
            callback_func (function (EnvRenderer) -> None): custom function to called during render()
        """

        F110Env.render_callbacks.append(callback_func)

    # def render(self, mode='human'):
    #     """
    #     Renders the environment with pyglet. Use mouse scroll in the window to zoom in/out, use mouse click drag to pan. Shows the agents, the map, current fps (bottom left corner), and the race information near as text.

    #     Args:
    #         mode (str, default='human'): rendering mode, currently supports:
    #             'human': slowed down rendering such that the env is rendered in a way that sim time elapsed is close to real time elapsed
    #             'human_fast': render as fast as possible

    #     Returns:
    #         None
    #     """
    #     assert mode in ['human', 'human_fast']
        
    #     if F110Env.renderer is None:
    #         # first call, initialize everything
    #         from f1tenth_gym.f110_gym.rendering import EnvRenderer
    #         F110Env.renderer = EnvRenderer(WINDOW_W, WINDOW_H)
    #         F110Env.renderer.update_map(self.map_name, self.map_ext)
            
    #     F110Env.renderer.update_obs(self.render_obs)

    #     for render_callback in F110Env.render_callbacks:
    #         render_callback(F110Env.renderer)
        
    #     F110Env.renderer.dispatch_events()
    #     F110Env.renderer.on_draw()
    #     F110Env.renderer.flip()
    #     if mode == 'human':
    #         time.sleep(0.005)
    #     elif mode == 'human_fast':
            # pass

    def render(self, mode='human'):
        """
        Renders the environment with pyglet. Use mouse scroll in the window to zoom in/out, use mouse click drag to pan. Shows the agents, the map, current fps (bottom left corner), and the race information near as text.

        Args:
            mode (str, default='human'): rendering mode, currently supports:
                'human': slowed down rendering such that the env is rendered in a way that sim time elapsed is close to real time elapsed
                'human_fast': render as fast as possible

        Returns:
            None
        """
        assert mode in ['human', 'human_fast']
        
        if self.renderer is None:
            # first call, initialize everything
            self.renderer = EnvRenderer(WINDOW_W, WINDOW_H)
            self.renderer.update_map(self.map_name, self.map_ext)
            
        self.renderer.update_obs(self.render_obs)

        for render_callback in self.render_callbacks:
            render_callback(self.renderer)
        
        self.renderer.dispatch_events()
        self.renderer.on_draw()
        self.renderer.flip()
        if mode == 'human':
            time.sleep(0.005)
        elif mode == 'human_fast':
            pass

    def close_rendering(self):
        if self.renderer is not None:
            self.renderer.close()        

    def save_traj(self, name="Traj"):
        poses = np.array(self.poses)
        plt.figure(1)
        plt.clf()
        plt.plot(poses[:,0], poses[:,1], 'b-')

        plt.gca().set_aspect('equal', adjustable='box')

        plt.pause(0.001)

        plt.savefig("Data/Trajectories/" + name + ".png")

        plt.show()
    
    def render_trajectory(self, path, save_name="VehicleName", safety_history=None):
        """
        Renders the map using the plt library

        Args:
            wait: plt.show() should be called or not
        """
        # self.env_map.render_map(4)
        # plt.show()
        fig = plt.figure(4)
        plt.clf()
        plt.title(save_name)

        # xs, ys = self.env_map.convert_positions(self.history.positions)
        poses = np.array(self.poses)
        poses = poses[0:-1:5]
        xs, ys = poses[:,0], poses[:,1]
        if safety_history is None:
            plt.plot(xs, ys, 'r', linewidth=3)
            # plt.plot(xs, ys, '+', markersize=12)
        else:
            N = len(safety_history.planned_actions)
            for i in range(N-1):
                x_pts = [xs[i], xs[i+1]]
                y_pts = [ys[i], ys[i+1]]
                if safety_history.planned_actions[i][0] == safety_history.safe_actions[i][0]:
                    plt.plot(x_pts, y_pts, 'r', linewidth=3)
                else:
                    plt.plot(x_pts, y_pts, 'b', linewidth=3)
                
        plt.gca().set_aspect('equal', adjustable='box')

        plt.savefig(f"{path}/{save_name}_track.svg")
        plt.savefig(f"{path}/{save_name}_track.png") # for easy viewing

        safety_history.planned_actions = []
        safety_history.safe_actions = []

In [ ]:
dataclass
class F1TenthEnvConfig:
    map_yaml: str
    map_ext: str
    timestep: float = 0.01
    control_hz: int = 20
    max_steer_rad: float = 0.4
    max_speed_mps: float = 8.0
    render: bool = False
    # Number of beams fed to the learned policy.  ``None`` uses the width the
    # loaded model was trained with.  Do not use a fixed stride here: F1TENTH
    # installations can expose different native scan resolutions.
    lidar_dim: Optional[int] = None
    action_repeat: int = 1


class F1TenthEnvironment:


    def __init__(self, model: OPOSMSkillModel, state_normalizer: Normalizer,
                 action_normalizer: Normalizer, cfg: F1TenthEnvConfig,
                 initial_state_raw: np.ndarray, device: str = "cpu"):
        if model is not None and state_normalizer is not None and action_normalizer is not None:
            self.model = model.to(device)
            self.model.eval()
            self.state_normalizer = state_normalizer
            self.action_normalizer = action_normalizer
        self.device = device
        self.cfg = cfg
        self.lidar_dim = cfg.lidar_dim or model.lidar_dim
        if self.lidar_dim <= 0:
            raise ValueError(f"lidar_dim must be positive, got {self.lidar_dim}")
        self.env = F110Env(map=cfg.map_yaml.split("/")[-1][:-9]) 
        self._substeps = max(1, round(1.0 / (cfg.control_hz * cfg.timestep)))

        self.crashed = False
        self.crash_step: Optional[int] = None
        self._step_count = 0
        self.reset(initial_state_raw)

    def _pose_for_reset(self, state_raw: np.ndarray) -> np.ndarray:
        x, y, yaw = state_raw[0], state_raw[1], state_raw[4] 
        return np.array([[x, y, yaw]], dtype=np.float64)

    def reset(self, state_raw: np.ndarray):
        self.obs, _, self.done, _ = self.env.reset(self._pose_for_reset(state_raw))
        self.crashed = False
        self.crash_step = None
        self._step_count = 0

    def get_state(self) -> np.ndarray:
        return np.asarray(self.env.sim.agents[0].state, dtype=np.float32).copy()

    def get_lidar(self) -> np.ndarray:
        return self._policy_scan().copy()

    def _raw_step(self, steer: float, speed: float):
        steer = float(np.clip(steer, -self.cfg.max_steer_rad, self.cfg.max_steer_rad))
        speed = float(np.clip(speed, 0.0, self.cfg.max_speed_mps))
        action = np.array([[steer, speed]], dtype=np.float64)
        self.obs, _, self.done, _ = self.env.step(action)
        self._step_count += 1
        if not self.crashed and bool(self.obs["collisions"][0]):
            self.crashed = True
            self.crash_step = self._step_count
            print(f"[f1tenth_env] WARNING: collision at control step {self._step_count}.")
        if self.cfg.render:
            self.env.render(mode="human")

    def step_raw_action(self, action_raw: np.ndarray) -> np.ndarray:
        for _ in range(self._substeps):
            self._raw_step(action_raw[0], action_raw[1])
        return self.get_state()

    def _policy_scan(self) -> np.ndarray:
        """Return exactly the LiDAR width expected by the learned policy.

        Native F1TENTH scan lengths vary by simulator/configuration (for
        example, 108 or 1080 beams).  Selecting every Nth beam can therefore
        silently change the input width.  Evenly sampling the complete scan
        preserves its field of view and keeps the network input stable.
        """
        full_scan = np.asarray(self.obs["scans"][0], dtype=np.float32).reshape(-1)
        if full_scan.size == 0:
            raise RuntimeError("F1TENTH returned an empty LiDAR scan")
        if full_scan.size == self.lidar_dim:
            return full_scan

        indices = np.linspace(0, full_scan.size - 1, self.lidar_dim).round().astype(np.intp)
        return full_scan[indices]

    def execute_skill(self, z: np.ndarray, horizon: int) -> np.ndarray:
        z_t = torch.as_tensor(z, dtype=torch.float32, device=self.device).unsqueeze(0)
        hidden = None
        for _ in range(horizon):
            scan = torch.as_tensor(self._policy_scan(), dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad():
                o_embed = self.model.lidar_encoder(scan)
                action_dist, hidden = self.model.low_level_policy.step(o_embed, z_t, hidden)
                a_norm = action_dist.mean[0].cpu().numpy()
            a_raw = self.action_normalizer.inverse_transform(a_norm[None, :])[0]
            for _ in range(self.cfg.action_repeat):
                for _ in range(self._substeps):
                    self._raw_step(a_raw[0], a_raw[1])
        return self.get_state()

    def close(self):
        pass

In [ ]:
@dataclass
class PurePursuitConfig:
    lookahead_m: float = 1.6
    min_lookahead_m: float = 0.8
    max_lookahead_m: float = 3.5
    lookahead_speed_gain: float = 0.15  
    wheelbase_m: float = 0.33
    max_steer_rad: float = 0.4
    target_speed_mps: float = 5.0
    corner_slowdown_gain: float = 2.5   
    min_speed_mps: float = 1.0


class PurePursuitController:
    def __init__(self, centerline: Centerline, cfg: PurePursuitConfig):
        self.centerline = centerline
        self.cfg = cfg

    def _lookahead_point(self, xy: np.ndarray, lookahead_m: float) -> np.ndarray:
        idx = self.centerline.nearest_index(xy)
        target_dist = self.centerline.cum_dist[idx] + lookahead_m
        target_dist = target_dist % self.centerline.total_length
        target_idx = int(np.searchsorted(self.centerline.cum_dist, target_dist, side="left"))
        target_idx = target_idx % self.centerline.num_points
        return self.centerline.points[target_idx]

    def act(self, x: float, y: float, yaw: float, speed: float) -> np.ndarray:
        """Returns raw [steer_cmd, speed_cmd] for the current raw pose."""
        cfg = self.cfg
        lookahead = float(np.clip(
            cfg.min_lookahead_m + cfg.lookahead_speed_gain * speed,
            cfg.min_lookahead_m, cfg.max_lookahead_m,
        ))
        target = self._lookahead_point(np.array([x, y], dtype=np.float32), lookahead)

        dx = target[0] - x
        dy = target[1] - y
        cos_yaw, sin_yaw = np.cos(-yaw), np.sin(-yaw)
        local_x = cos_yaw * dx - sin_yaw * dy
        local_y = sin_yaw * dx + cos_yaw * dy

        curvature = 2.0 * local_y / max(lookahead ** 2, 1e-6)
        steer_cmd = np.arctan(cfg.wheelbase_m * curvature)
        steer_cmd = float(np.clip(steer_cmd, -cfg.max_steer_rad, cfg.max_steer_rad))

        speed_cmd = cfg.target_speed_mps - cfg.corner_slowdown_gain * abs(steer_cmd)
        speed_cmd = float(max(speed_cmd, cfg.min_speed_mps))

        return np.array([steer_cmd, speed_cmd], dtype=np.float32)


def add_exploration_noise(action: np.ndarray, rng: np.random.RandomState,
                           steer_std: float = 0.03, speed_std: float = 0.3,
                           max_steer_rad: float = 0.4) -> np.ndarray:
    noisy = action.copy()
    noisy[0] = float(np.clip(action[0] + rng.normal(0, steer_std), -max_steer_rad, max_steer_rad))
    noisy[1] = max(0.0, action[1] + rng.normal(0, speed_std))
    return noisy

In [ ]:
from __future__ import annotations

import glob
import os
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from utils.normalization import Normalizer
from utils.track import Centerline


@dataclass
class RunArrays:
    name: str
    map_name: str
    states: np.ndarray
    actions: np.ndarray
    raw_states: np.ndarray
    lidar: np.ndarray
    dones: np.ndarray
    collisions: np.ndarray


def discover_runs(root: str, explicit: Optional[List[str]] = None) -> List[str]:
    if explicit:
        return explicit
    files = sorted(glob.glob(os.path.join(root, "*", "run-*.csv")))
    if not files:
        raise FileNotFoundError(
            f"No 'run-*.csv' files found under '{root}'. Run "
            f"scripts/generate_offline_data.py first to collect the offline dataset."
        )
    return [os.path.relpath(f, root) for f in files]


def load_run_dataframe(root: str, run_rel_path: str, dropna: bool, stride: int) -> pd.DataFrame:
    df = pd.read_csv(os.path.join(root, run_rel_path))
    if dropna:
        df = df.dropna().reset_index(drop=True)
    if stride > 1:
        df = df.iloc[::stride].reset_index(drop=True)
    return df


def lidar_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if c.startswith("lidar_")]
    return sorted(cols, key=lambda c: int(c.split("_")[1]))


class OfflineSkillDataset(Dataset):
    def __init__(self, runs: List[RunArrays], window_len: int, drop_collision_windows: bool = False):
        """
        Args:
            runs: per-episode arrays (see build_run_arrays).
            window_len: number of states per window (window_len - 1 actions).
            drop_collision_windows: if True, skip any window whose terminal
                transition (the action landing on sT) ended in a collision.
                Useful if you want a world model trained only on "clean"
                dynamics, with collision transitions handled separately.
        """
        assert window_len >= 2
        self.window_len = window_len
        self.runs = runs

        self._index: List[Tuple[int, int]] = []
        for ri, run in enumerate(runs):
            n = run.states.shape[0]
            last_start = n - window_len
            if last_start < 0:
                continue
            for t in range(last_start + 1):
                terminal_idx = t + window_len - 2  # row whose action produced sT
                if drop_collision_windows and bool(run.collisions[terminal_idx]):
                    continue
                self._index.append((ri, t))

        if not self._index:
            raise ValueError(
                "No run produced a single valid sub-trajectory window -- "
                "reduce data.subtraj_len/subsample_stride, disable "
                "drop_collision_windows, or collect longer/more runs with "
                "scripts/generate_offline_data.py."
            )

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx):
        ri, t = self._index[idx]
        run = self.runs[ri]
        T = self.window_len - 1
        states_window = run.states[t: t + self.window_len]
        actions_window = run.actions[t: t + T]
        lidar_window = run.lidar[t: t + T]
        terminal_idx = t + T - 1
        return {
            "s0": torch.as_tensor(states_window[0], dtype=torch.float32),
            "states_seq": torch.as_tensor(states_window[:-1], dtype=torch.float32),
            "actions_seq": torch.as_tensor(actions_window, dtype=torch.float32),
            "lidar_seq": torch.as_tensor(lidar_window, dtype=torch.float32),
            "lidar_T": torch.as_tensor(run.lidar[t + T], dtype=torch.float32),
            "sT": torch.as_tensor(states_window[-1], dtype=torch.float32),
            "done": torch.tensor(bool(run.dones[terminal_idx]), dtype=torch.bool),
            "collision": torch.tensor(bool(run.collisions[terminal_idx]), dtype=torch.bool),
        }


def build_run_arrays(
    root: str, run_names: List[str], state_features: List[str], action_features: List[str],
    dropna: bool, stride: int,
    state_normalizer: Optional[Normalizer] = None, action_normalizer: Optional[Normalizer] = None,
) -> Tuple[List[RunArrays], Normalizer, Normalizer]:

    raw_states_per_run, raw_actions_per_run, lidar_per_run, map_names, names = [], [], [], [], []
    dones_per_run, collisions_per_run = [], []
    for name in run_names:
        df = load_run_dataframe(root, name, dropna, stride)
        missing_state = [c for c in state_features if c not in df.columns]
        missing_action = [c for c in action_features if c not in df.columns]
        if missing_state or missing_action:
            raise ValueError(
                f"Run '{name}' is missing expected columns. "
                f"missing state cols={missing_state}, missing action cols={missing_action}"
            )
        lidar_cols = lidar_columns(df)
        if not lidar_cols:
            raise ValueError(
                f"Run '{name}' has no 'lidar_*' columns. "
                f"Re-run scripts/generate_offline_data.py with the updated collector."
            )
        raw_states_per_run.append(df[state_features].to_numpy(dtype=np.float32))
        raw_actions_per_run.append(df[action_features].to_numpy(dtype=np.float32))
        lidar_per_run.append(df[lidar_cols].to_numpy(dtype=np.float32))
        map_names.append(df["map_name"].iloc[0] if "map_name" in df.columns else "unknown")
        names.append(name)

        n = len(df)
        if "done" in df.columns:
            dones = df["done"].to_numpy(dtype=bool)
        else:
            dones = np.zeros(n, dtype=bool)
            if n > 0:
                dones[-1] = True
        if "collision" in df.columns:
            collisions = df["collision"].to_numpy(dtype=bool)
        else:
            collisions = np.zeros(n, dtype=bool)
        dones_per_run.append(dones)
        collisions_per_run.append(collisions)

    all_raw_states = np.concatenate(raw_states_per_run, axis=0)
    all_raw_actions = np.concatenate(raw_actions_per_run, axis=0)

    if state_normalizer is None:
        state_normalizer = Normalizer.fit(all_raw_states, feature_names=state_features)
    if action_normalizer is None:
        action_normalizer = Normalizer.fit(all_raw_actions, feature_names=action_features)

    runs = []
    for name, map_name, raw_s, raw_a, lidar, dones, collisions in zip(
        names, map_names, raw_states_per_run, raw_actions_per_run, lidar_per_run,
        dones_per_run, collisions_per_run,
    ):
        runs.append(RunArrays(
            name=name, map_name=map_name,
            states=state_normalizer.transform(raw_s).astype(np.float32),
            actions=action_normalizer.transform(raw_a).astype(np.float32),
            raw_states=raw_s,
            lidar=lidar,
            dones=dones,
            collisions=collisions,
        ))
    return runs, state_normalizer, action_normalizer


def train_test_run_split(run_names: List[str], train_frac: float, seed: int = 0):
    rng = np.random.RandomState(seed)
    names = list(run_names)
    rng.shuffle(names)
    n_train = max(1, int(round(train_frac * len(names))))
    return names[:n_train], names[n_train:]


def load_centerline_for_run(maps_root: str, map_name:str) -> Centerline:
    from utils.track import discover_map
    map_yaml,map_ext, centerline_csv = discover_map(maps_root, map_name)
    return map_yaml , map_ext ,Centerline.from_csv(centerline_csv)


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Optional

import torch
from torch.utils.data import DataLoader

from models.skill_model import OPOSMSkillModel
from utils.logger import Logger

@dataclass
class EMTrainerConfig:
    lr_e: float = 5e-5
    lr_m: float = 5e-5
    grad_clip_norm: float = 10.0
    device: str = "cuda"
    checkpoint_dir: str = "checkpoints"
    log_every: int = 20
    save_every_epochs: int = 10
    mode: str = "causal_em"
    aux_loss_weight: float = 1.0
    metrics_path: Optional[str] = None  # defaults to <checkpoint_dir>/metrics.json


class EMTrainer:
    def __init__(self, model: OPOSMSkillModel, cfg: EMTrainerConfig):
        self.cfg = cfg
        self.device = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        phi_params = list(self.model.skill_posterior.parameters())
        theta_psi_omega_params = (
            list(self.model.low_level_policy.parameters())
            + list(self.model.tawm.parameters())
            + list(self.model.skill_prior.parameters())
            + list(self.model.lidar_encoder.parameters())  # encoder is shared, trained only via the M-step objective
        )

        if cfg.mode == "causal_em":
            self.opt_e = torch.optim.Adam(phi_params, lr=cfg.lr_e)
            self.opt_m = torch.optim.Adam(theta_psi_omega_params, lr=cfg.lr_m)
        elif cfg.mode == "naive_vi":
            self.opt_joint = torch.optim.Adam(
                phi_params + theta_psi_omega_params, lr=cfg.lr_m
            )
        else:
            raise ValueError(f"Unknown training mode: {cfg.mode}")

        self.logger = Logger(log_every=cfg.log_every)
        os.makedirs(cfg.checkpoint_dir, exist_ok=True)
        self.global_step = 0

        # --- metrics history, for later plotting ---
        self.metrics_path = cfg.metrics_path or os.path.join(cfg.checkpoint_dir, "metrics.json")
        self.metrics_history: list[dict] = []
        self.validation_metrics_path = os.path.join(cfg.checkpoint_dir, "validation_metrics.json")
        self.validation_history: list[dict] = []

    def _to_device(self, batch):
        return {k: v.to(self.device) for k, v in batch.items()}

    def train_step(self, batch, epoch: Optional[int] = None) -> dict:
        batch = self._to_device(batch)

        if self.cfg.mode == "causal_em":
            self.opt_e.zero_grad(set_to_none=True)
            self.model.lidar_encoder.zero_grad(set_to_none=True)  # encoder isn't in opt_e; zero it so e-step grads don't leak into the m-step update below
            e_out = self.model.e_loss_causal(batch)
            e_out["loss"].backward()
            torch.nn.utils.clip_grad_norm_(self.model.skill_posterior.parameters(),
                                            self.cfg.grad_clip_norm)
            self.opt_e.step()

            self.opt_m.zero_grad(set_to_none=True)
            m_out = self.model.m_loss(batch)
            m_out["loss"].backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.model.low_level_policy.parameters())
                + list(self.model.tawm.parameters())
                + list(self.model.skill_prior.parameters())
                + list(self.model.lidar_encoder.parameters()),
                self.cfg.grad_clip_norm,
            )
            self.opt_m.step()

            metrics = {
                "e_loss": e_out["loss"].item(),
                "m_loss": m_out["loss"].item(),
                "elbo": m_out["elbo"].item(),
                "log_pi": m_out["log_pi"].item(),
                "log_psi": m_out["log_psi"].item(),
                "kl": m_out["kl"].item(),
            }
        else:
            self.opt_joint.zero_grad(set_to_none=True)
            out = self.model.naive_vi_e_loss(batch)
            out["loss"].backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
            self.opt_joint.step()
            metrics = {"joint_loss": out["loss"].item(), "elbo": out["elbo"].item()}

        self.global_step += 1
        self.logger.log(self.global_step, **metrics)

        # --- record + periodically persist metrics for plotting ---
        record = {"step": self.global_step, "epoch": epoch, **metrics}
        self.metrics_history.append(record)
        if self.global_step % self.cfg.log_every == 0:
            self._save_metrics()

        return metrics

    @torch.no_grad()
    def evaluate(self, dataloader: DataLoader) -> dict[str, float]:
        """Aggregate held-out ELBO terms and terminal-scan reconstruction MSE."""
        was_training = self.model.training
        self.model.eval()
        totals = {key: 0.0 for key in ("loss", "elbo", "log_pi", "log_psi", "kl", "lidar_mse")}
        count = 0
        for batch in dataloader:
            batch = self._to_device(batch)
            terms = self.model.elbo_terms(batch)
            target = torch.clamp(
                batch["lidar_T"], min=0.0, max=self.model.lidar_max_range
            ) / self.model.lidar_max_range
            o0_embed = self.model.lidar_encoder(batch["lidar_seq"][:, 0])
            pred, _ = self.model.tawm(o0_embed, terms["z"])
            batch_size = batch["lidar_T"].shape[0]
            values = {
                "loss": -terms["elbo"].mean(),
                "elbo": terms["elbo"].mean(),
                "log_pi": terms["log_pi"].mean(),
                "log_psi": terms["log_psi"].mean(),
                "kl": terms["kl"].mean(),
                "lidar_mse": (pred - target).square().mean(),
            }
            for key, value in values.items():
                totals[key] += float(value) * batch_size
            count += batch_size
        if was_training:
            self.model.train()
        if count == 0:
            raise ValueError("Validation dataloader is empty")
        return {key: value / count for key, value in totals.items()}

    def fit(self, dataloader: DataLoader, num_epochs: int, steps_per_epoch: Optional[int] = None,
            val_dataloader: Optional[DataLoader] = None):
        data_iter = iter(dataloader)
        for epoch in range(1, num_epochs + 1):
            steps = steps_per_epoch or len(dataloader)
            for _ in range(steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataloader)
                    batch = next(data_iter)
                self.train_step(batch, epoch=epoch)

            if val_dataloader is not None:
                val_metrics = self.evaluate(val_dataloader)
                val_record = {
                    "step": self.global_step,
                    "epoch": epoch,
                    **{f"val_{key}": value for key, value in val_metrics.items()},
                }
                self.validation_history.append(val_record)
                print(
                    f"[validation epoch {epoch:>4d}] "
                    + "  ".join(f"{key}={value:.4f}" for key, value in val_metrics.items())
                )
                self._save_validation_metrics()

            if epoch % self.cfg.save_every_epochs == 0 or epoch == num_epochs:
                self.save_checkpoint(os.path.join(self.cfg.checkpoint_dir, f"oposm_epoch{epoch}.pt"))

        self.save_checkpoint(os.path.join(self.cfg.checkpoint_dir, "oposm_final.pt"))
        self._save_metrics()  # final flush, guarantees everything is on disk

    def save_checkpoint(self, path: str):
        torch.save({
            "model_state_dict": self.model.state_dict(),
            "mode": self.cfg.mode,
            "global_step": self.global_step,
            "state_dim": self.model.state_dim,
            "action_dim": self.model.action_dim,
            "z_dim": self.model.z_dim,
            "lidar_dim": self.model.lidar_dim,
            "world_model_target": "terminal_lidar",
            "obs_embed_dim": self.model.obs_embed_dim,
            "tawm_predict_delta": self.model.tawm_predict_delta,
        }, path)
        print(f"[checkpoint] saved -> {path}")

    def _save_metrics(self, path: Optional[str] = None):
        path = path or self.metrics_path
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w") as f:
            json.dump(self.metrics_history, f, indent=2)

    def _save_validation_metrics(self):
        with open(self.validation_metrics_path, "w") as f:
            json.dump(self.validation_history, f, indent=2)

    @staticmethod
    def load_model(model: OPOSMSkillModel, path: str, device: str = "cpu") -> OPOSMSkillModel:
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        return model


In [ ]:
""" 
From https://github.com/BDEvan5/TrajectoryAidedLearning/blob/master/TrajectoryAidedLearning/Utils/RewardSignals.py wtih some modifications 
"""
from __future__ import annotations

from typing import Tuple, Optional

import numpy as np
import math, cmath




def get_distance(x1=(0, 0), x2=(0, 0)):
    d = [0.0, 0.0]
    for i in range(2):
        d[i] = x1[i] - x2[i]
    return np.linalg.norm(d)


def find_closest_pt(pt, wpts):
    """Returns the two closest points in order along wpts"""
    dists = [get_distance(pt, wpt) for wpt in wpts]
    min_i = np.argmin(dists)
    d_i = dists[min_i]
    if min_i == len(dists) - 1:
        min_i -= 1
    if dists[max(min_i - 1, 0)] > dists[min_i + 1]:
        p_i = wpts[min_i]
        p_ii = wpts[min_i + 1]
        d_i = dists[min_i]
        d_ii = dists[min_i + 1]
    else:
        p_i = wpts[min_i - 1]
        p_ii = wpts[min_i]
        d_i = dists[min_i - 1]
        d_ii = dists[min_i]
    return p_i, p_ii, d_i, d_ii


def get_tiangle_h(a, b, c):
    s = (a + b + c) / 2
    A = np.sqrt(s * (s - a) * (s - b) * (s - c))
    h = 2 * A / c
    return h


def get_gradient(x1=(0, 0), x2=(0, 0)):
    t = (x1[1] - x2[1])
    b = (x1[0] - x2[0])
    if b != 0:
        return t / b
    return 1000000  # near infinite gradient


def get_bearing(x1=(0, 0), x2=(0, 0)):
    grad = get_gradient(x1, x2)
    dx = x2[0] - x1[0]
    th_start_end = np.arctan(grad)
    if dx == 0:
        if x2[1] - x1[1] > 0:
            th_start_end = 0
        else:
            th_start_end = np.pi
    elif th_start_end > 0:
        if dx > 0:
            th_start_end = np.pi / 2 - th_start_end
        else:
            th_start_end = -np.pi / 2 - th_start_end
    else:
        if dx > 0:
            th_start_end = np.pi / 2 - th_start_end
        else:
            th_start_end = -np.pi / 2 - th_start_end
    return th_start_end


def robust_angle_difference_rad(x, y):
    """r = x - y, wrapped to [-pi, pi]"""
    return np.arctan2(np.sin(x - y), np.cos(x - y))

def get_cross_track_heading(point, wpts):
    p_i, p_ii, d_i, d_ii = find_closest_pt(point, wpts)
    d_ii_i = get_distance(p_i, p_ii)
    if d_ii_i < 1e-8:
        h = 0.0
    else:
        try:
            h = get_tiangle_h(d_i, d_ii, d_ii_i)
        except (ValueError, FloatingPointError):
            h = min(d_i, d_ii)
    heading = get_bearing(p_i, p_ii)
    return heading, h


def calculate_progress(point, wpts, cum_dist):
    """Closest-point arclength progress"""
    dists = [get_distance(point, wpt) for wpt in wpts]
    min_i = int(np.argmin(dists))
    if min_i == len(wpts) - 1:
        min_i -= 1
    p_i, p_ii = wpts[min_i], wpts[min_i + 1]
    seg_vec = np.array(p_ii) - np.array(p_i)
    seg_len = np.linalg.norm(seg_vec) + 1e-8
    t = np.clip(np.dot(np.array(point) - np.array(p_i), seg_vec) / (seg_len ** 2), 0.0, 1.0)
    return cum_dist[min_i] + t * seg_len


class LidarStructuralReward:
    """Map-free reward over predicted terminal scans shaped (K, L, beams)."""

    def __init__(self, lidar_dim: int, forward_weight: float = 1.0,
                 symmetry_weight: float = 0.2, collision_penalty: float = 100.0,
                 safe_distance: float = 0.35, forward_fraction: float = 0.30):
        if lidar_dim < 4:
            raise ValueError("LidarStructuralReward requires at least four beams")
        if not 0.0 < forward_fraction < 1.0:
            raise ValueError("forward_fraction must be between 0 and 1")
        self.lidar_dim = int(lidar_dim)
        self.forward_weight = float(forward_weight)
        self.symmetry_weight = float(symmetry_weight)
        self.collision_penalty = float(collision_penalty)
        self.safe_distance = float(safe_distance)
        self.reward_mode = "lidar_structural"

        forward_count = max(1, int(round(lidar_dim * forward_fraction)))
        start = (lidar_dim - forward_count) // 2
        self.forward_slice = slice(start, start + forward_count)
        side_end = start
        side_start = start + forward_count
        self.left_slice = slice(0, side_end)
        self.right_slice = slice(side_start, lidar_dim)
        if side_end == 0 or side_start == lidar_dim:
            raise ValueError("LiDAR scan must leave beams on both sides of the forward sector")

    def __call__(self, lidar_rollouts: np.ndarray) -> np.ndarray:
        scans = np.asarray(lidar_rollouts, dtype=np.float32)
        if scans.ndim != 3 or scans.shape[-1] != self.lidar_dim:
            raise ValueError(f"expected scans shaped (K, L, {self.lidar_dim}), got {scans.shape}")
        scans = np.nan_to_num(scans, nan=0.0, posinf=0.0, neginf=0.0)
        clearance = scans[..., self.forward_slice].mean(axis=-1)
        left = scans[..., self.left_slice].mean(axis=-1)
        right = scans[..., self.right_slice].mean(axis=-1)
        asymmetry = np.abs(left - right)
        collision = np.min(scans, axis=-1) < self.safe_distance

        per_skill = self.forward_weight * clearance - self.symmetry_weight * asymmetry
        active = np.cumsum(collision, axis=1) == 0
        rewards = (per_skill * active).sum(axis=1)
        rewards -= self.collision_penalty * collision.any(axis=1)
        return rewards.astype(np.float32)


class CenterlineProgressReward:
    def __init__(self, waypoints_xy: np.ndarray,
                 waypoint_bonus: float = 0.0, progress_weight: float = 1.0,
                 track_half_width: float = None, off_track_penalty: float = 0.0,
                 reward_mode: str = "progress", max_v: float = 1.0,
                 backward_penalty: float = 2.0,
                 max_progress_per_step: float = 12.0,
                 collision_penalty: float = 10.0):
        self.waypoints = waypoints_xy
        self.num_waypoints = waypoints_xy.shape[0]
        self.waypoint_bonus = waypoint_bonus
        self.progress_weight = progress_weight
        self.track_half_width = track_half_width
        self.off_track_penalty = off_track_penalty
        self.reward_mode = reward_mode
        self.max_v = max_v
        self.backward_penalty = backward_penalty
        self.max_progress_per_step = max_progress_per_step
        self.collision_penalty = collision_penalty

        if self.num_waypoints < 3:
            raise ValueError("CenterlineProgressReward needs at least three waypoints")

        # Treat the track as closed.  The final waypoint normally does not
        # duplicate the first, so the closing segment must be included.
        wp = self.waypoints
        self.segment_start = wp
        self.segment_vec = np.roll(wp, -1, axis=0) - wp
        self.segment_len = np.linalg.norm(self.segment_vec, axis=1)
        if np.any(self.segment_len < 1e-8):
            raise ValueError("Centerline contains duplicate adjacent waypoints")
        self.segment_len_sq = self.segment_len ** 2
        self.cum_dist = np.concatenate([[0.0], np.cumsum(self.segment_len)])[:-1]
        self.total_s = float(np.sum(self.segment_len))

        # TALearningReward constants, taken directly from your snippet
        self.beta_c = 0.4
        self.beta_steer_weight = 0.4
        self.beta_velocity_weight = 0.4
        self.max_steer_diff = 0.8
        self.max_velocity_diff = 2.0

        # CrossTrackHeadReward constants, taken directly from your snippet
        self.r_velocity = 1
        self.r_distance = 1

    def _project_positions(self, positions: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Project ``(..., 2)`` positions onto the closed centerline.

        This is deliberately vectorized: CEM scores thousands of trajectories
        every iteration, and the former per-point Python nearest-waypoint loop
        dominated planning time on CPU.
        """
        points = np.asarray(positions, dtype=np.float32)
        offset = points[..., None, :] - self.segment_start
        t = np.sum(offset * self.segment_vec, axis=-1) / self.segment_len_sq
        t = np.clip(t, 0.0, 1.0)
        projected = self.segment_start + t[..., None] * self.segment_vec
        d2 = np.sum((points[..., None, :] - projected) ** 2, axis=-1)
        segment_idx = np.argmin(d2, axis=-1)
        chosen_t = np.take_along_axis(t, segment_idx[..., None], axis=-1)[..., 0]
        s = self.cum_dist[segment_idx] + chosen_t * self.segment_len[segment_idx]
        cross_track_error = np.sqrt(np.take_along_axis(d2, segment_idx[..., None], axis=-1)[..., 0])
        return s, cross_track_error

    def __call__(self, positions: np.ndarray,
                  headings: Optional[np.ndarray] = None,
                  velocities: Optional[np.ndarray] = None,
                  agent_actions: Optional[np.ndarray] = None,
                  classic_actions: Optional[np.ndarray] = None,
                  collisions: Optional[np.ndarray] = None,
                  lap_done: Optional[np.ndarray] = None,
                  init_position: Optional[np.ndarray] = None
                  ) -> Tuple[np.ndarray, np.ndarray]:
        """
        positions:       (K, L, 2)
        headings:        (K, L)      required for "cth"
        velocities:      (K, L)      required for "cth"
        agent_actions:   (K, L, 2)   required for "tal"  [steer, speed]
        classic_actions: (K, L, 2)   required for "tal"  [steer, speed] from planner
        collisions:      (K, L) bool optional, terminal -1 (all modes)
        lap_done:        (K, L) bool optional, terminal +1 (all modes)
        init_position:   (K, 2) optional, position just before l=0, needed
                          for "progress" mode's first-step delta
        """
        K, L, _ = positions.shape
        rewards = np.zeros(K, dtype=np.float32)
        wpts_list = self.waypoints.tolist()

        if self.reward_mode == "progress":
            s, cross_track_error = self._project_positions(positions)
            if init_position is None:
                prev_s = s[:, :1]
            else:
                prev_s, _ = self._project_positions(init_position)
                prev_s = prev_s[:, None]
            delta_s = np.diff(np.concatenate([prev_s, s], axis=1), axis=1)
            delta_s = np.where(delta_s < -0.5 * self.total_s, delta_s + self.total_s, delta_s)
            delta_s = np.where(delta_s > 0.5 * self.total_s, delta_s - self.total_s, delta_s)
            forward = np.clip(delta_s, 0.0, self.max_progress_per_step)
            backward = np.clip(-delta_s, 0.0, self.max_progress_per_step)
            valid = np.ones((K, L), dtype=bool)
            if collisions is not None:
                collisions = np.asarray(collisions, dtype=bool)
                if collisions.shape != (K, L):
                    raise ValueError(f"collisions must have shape {(K, L)}, got {collisions.shape}")
                valid = np.cumsum(collisions, axis=1) == 0

            rewards = self.progress_weight * (forward * valid).sum(axis=1)
            rewards -= self.backward_penalty * (backward * valid).sum(axis=1)
            if self.track_half_width is not None and self.off_track_penalty:
                excess = np.clip(cross_track_error - self.track_half_width, 0.0, None)
                rewards -= self.off_track_penalty * (np.square(excess) * valid).sum(axis=1)
            if collisions is not None:
                rewards -= self.collision_penalty * collisions.any(axis=1)
            return rewards.astype(np.float32)

        for k in range(K):
            r = 0.0
            prev_pos = init_position[k] if init_position is not None else positions[k, 0]

            for l in range(L):
                pos = positions[k, l]

                if collisions is not None and collisions[k, l]:
                    r -= self.collision_penalty
                    break
                if lap_done is not None and lap_done[k, l]:
                    r += 1
                    break

                if self.reward_mode == "cth":
                    if headings is None or velocities is None:
                        raise ValueError("headings and velocities are required for reward_mode='cth'")
                    theta = float(headings[k, l])
                    v = float(velocities[k, l])
                    track_heading, dc = get_cross_track_heading(pos, wpts_list)
                    d_heading = abs(robust_angle_difference_rad(track_heading, theta))
                    r_heading = np.cos(d_heading) * self.r_velocity
                    r_heading *= (v / self.max_v)
                    r_dist = dc * self.r_distance
                    step_r = r_heading - r_dist
                    step_r = max(step_r, 0)  # from source

                elif self.reward_mode == "tal":
                    if agent_actions is None or classic_actions is None:
                        raise ValueError("agent_actions and classic_actions required for reward_mode='tal'")
                    u_agent = agent_actions[k, l]
                    u_classic = classic_actions[k, l]
                    steer_r = (abs(u_classic[0] - u_agent[0]) / self.max_steer_diff) * self.beta_steer_weight
                    throttle_r = (abs(u_classic[1] - u_agent[1]) / self.max_velocity_diff) * self.beta_velocity_weight
                    step_r = self.beta_c - steer_r - throttle_r
                    step_r = max(step_r, 0)  
                    step_r *= 0.5  

                else:
                    raise ValueError(f"Unknown reward_mode: {self.reward_mode}")

                r += self.progress_weight * step_r
                prev_pos = pos

            rewards[k] = r


        return rewards


In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Callable, List, Optional, Protocol

import numpy as np
import torch

from models.skill_model import OPOSMSkillModel
from planning.reward import LidarStructuralReward, calculate_progress, get_distance


@dataclass
class CEMPlannerConfig:
    plan_length: int = 8
    population_size: int = 500
    elite_frac: float = 0.10
    cem_iters: int = 10
    init_std: float = 1.0
    device: str = "cuda"
    x_idx: int = 0
    y_idx: int = 1
    yaw_idx: Optional[int] = 4
    v_idx: Optional[int] = 3

    # --- NEW: imagined-rollout saving ---
    save_dir: Optional[str] = "outputs/track"       # if set, plan() dumps an .npz per call here
    save_population: bool = True         # save the full last-iteration population, not just the best
    forward_reward_weight: float = 1.0
    symmetry_penalty_weight: float = 0.2
    lidar_collision_penalty: float = 100.0
    safe_distance_m: float = 0.35
    forward_beam_fraction: float = 0.30

    @property
    def num_elite(self) -> int:
        return max(1, int(round(self.population_size * self.elite_frac)))


RewardFn = Callable[..., np.ndarray]


class Environment(Protocol):

    def get_state(self) -> np.ndarray: ...
    def get_lidar(self) -> np.ndarray: ...

    def execute_skill(self, z: np.ndarray, horizon: int) -> np.ndarray:
        """Executes skill z for `horizon` low-level control ticks and
        returns the resulting raw state."""
        ...


class CEMPlanner:
    def __init__(self, model: OPOSMSkillModel,
                 reward_fn: LidarStructuralReward, cfg: CEMPlannerConfig,
                 collision_checker: Optional[Callable[[np.ndarray], np.ndarray]] = None):
        self.model = model
        self.reward_fn = reward_fn
        self.cfg = cfg
        self.collision_checker = collision_checker
        self.device = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

        if cfg.save_dir is not None:
            os.makedirs(cfg.save_dir, exist_ok=True)

    def _rollout_population(self, o0: torch.Tensor, epsilons: torch.Tensor) -> torch.Tensor:
        K, L, _ = epsilons.shape
        lidar = o0.expand(K, -1).clone()
        predicted_lidar = []
        with torch.no_grad():
            for l in range(L):
                lidar_embed = self.model.lidar_encoder(lidar)
                mu0, sigma0 = self.model.skill_prior(lidar_embed)
                z = mu0 + sigma0 * epsilons[:, l, :]
                mu_next, _ = self.model.tawm(lidar_embed, z)
                lidar = mu_next.clamp(0.0, 1.0) * self.model.lidar_max_range
                predicted_lidar.append(lidar)
        return torch.stack(predicted_lidar, dim=1)

    def plan(self, o0_raw: np.ndarray, step_idx: Optional[int] = None) -> dict:
        cfg = self.cfg
        z_dim = self.model.z_dim

        o0 = torch.as_tensor(o0_raw, dtype=torch.float32, device=self.device).reshape(1, -1)

        mean = torch.zeros(cfg.plan_length, z_dim, device=self.device)
        std = torch.full((cfg.plan_length, z_dim), cfg.init_std, device=self.device)

        best_epsilon, best_reward, best_predicted_lidar = None, -np.inf, None

        # kept from the LAST cem_iter only, for visualization of the final population
        last_population_lidar, last_rewards, last_elite_idx = None, None, None

        for _ in range(cfg.cem_iters):
            eps = mean.unsqueeze(0) + std.unsqueeze(0) * torch.randn(
                cfg.population_size, cfg.plan_length, z_dim, device=self.device
            )

            predicted_lidar = self._rollout_population(o0, eps)
            predicted_lidar_np = predicted_lidar.detach().cpu().numpy()
            rewards = self.reward_fn(predicted_lidar_np)

            elite_idx = np.argsort(-rewards)[: cfg.num_elite]
            elite_eps = eps[elite_idx]
            mean = elite_eps.mean(dim=0)
            std = elite_eps.std(dim=0).clamp_min(1e-3)

            top1 = elite_idx[0]
            if rewards[top1] > best_reward:
                best_reward = float(rewards[top1])
                best_epsilon = eps[top1].detach().clone()
                best_predicted_lidar = predicted_lidar[top1].detach().clone()

            # NEW: remember this iteration's population -- overwritten each loop,
            # so after the loop these hold the LAST iteration's data
            last_population_lidar = predicted_lidar_np
            last_rewards = rewards
            last_elite_idx = elite_idx

        print(f"[CEMPlanner] best reward: {best_reward:.3f}")

        # NEW: dump the imagined rollouts for this planning call
        if cfg.save_dir is not None:
            self._save_planning_step(
                step_idx=step_idx,
                population_lidar=last_population_lidar,
                population_rewards=last_rewards,
                elite_idx=last_elite_idx,
                best_predicted_lidar=best_predicted_lidar.cpu().numpy(),
                best_reward=best_reward,
            )

        return {
            "epsilon_seq": best_epsilon.cpu().numpy(),
            "predicted_lidar": best_predicted_lidar.cpu().numpy(),
            "reward": best_reward,
            "predicted_collisions": (best_predicted_lidar.min(dim=-1).values.cpu().numpy() < self.reward_fn.safe_distance),
        }

    def _save_planning_step(self, step_idx, population_lidar, population_rewards,
                             elite_idx, best_predicted_lidar, best_reward) -> None:
        """Save one planning call's imagined rollouts to cfg.save_dir/plan_step_XXXX.npz."""
        cfg = self.cfg
        fname = f"plan_step_{step_idx:05d}.npz" if step_idx is not None else "plan_latest.npz"
        path = os.path.join(cfg.save_dir, fname)

        save_kwargs = dict(
            elite_idx=elite_idx,
            best_predicted_lidar=best_predicted_lidar,   # (L, beams)
            best_reward=np.float32(best_reward),
        )
        if cfg.save_population:
            save_kwargs["population_lidar"] = population_lidar.astype(np.float32)  # (K, L, beams)
            save_kwargs["population_rewards"] = population_rewards.astype(np.float32)       # (K,)

        np.savez_compressed(path, **save_kwargs)

    @torch.no_grad()
    def epsilon_to_z(self, o_raw: np.ndarray, epsilon: np.ndarray) -> np.ndarray:
        o_embed = self.model.lidar_encoder(torch.as_tensor(o_raw, dtype=torch.float32, device=self.device).reshape(1, -1))
        eps_t = torch.as_tensor(epsilon, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            mu0, sigma0 = self.model.skill_prior(o_embed)
            z = mu0 + sigma0 * eps_t
        return z.squeeze(0).cpu().numpy()


def calculate_progress_local(point, wpts, cum_dist, prev_i=None, window=15):
    n = len(wpts)
    if prev_i is None:
        search_idx = range(n)
    else:
        search_idx = [i % n for i in range(prev_i - window, prev_i + window)]
    dists = [get_distance(point, wpts[i]) for i in search_idx]
    min_local = int(np.argmin(dists))
    min_i = search_idx[min_local]
    if min_i == n - 1:
        min_i -= 1
    p_i, p_ii = wpts[min_i], wpts[min_i + 1]
    seg_vec = np.array(p_ii) - np.array(p_i)
    seg_len = np.linalg.norm(seg_vec) + 1e-8
    t = np.clip(np.dot(np.array(point) - np.array(p_i), seg_vec) / (seg_len ** 2), 0.0, 1.0)
    s = cum_dist[min_i] + t * seg_len
    return s, min_i


def mpc_race(planner: CEMPlanner, env: Environment, waypoints_raw: np.ndarray,
             skill_horizon: int, max_skills: int = 200, max_laps: int = 1,
             replan_every_skills: int = 1, save_dir: Optional[str] = None) -> dict:
    trajectory = [env.get_state().copy()]
    executed_zs: List[np.ndarray] = []

    cfg = planner.cfg
    wpts_list = waypoints_raw.tolist()
    seg_len = np.linalg.norm(np.diff(waypoints_raw, axis=0), axis=1)
    cum_dist = np.concatenate([[0.0], np.cumsum(seg_len)])
    total_s = float(cum_dist[-1]) + 1e-8

    # NEW: wire up saving for this race
    if save_dir is not None:
        planner.cfg.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        np.savez_compressed(os.path.join(save_dir, "track.npz"), waypoints=waypoints_raw)

    laps_completed = 0
    stopped_for_predicted_collision = False
    collision_occurred = False  
    plan_call_idx = 0  

    prev_s, prev_i = calculate_progress_local(
        env.get_state()[[cfg.x_idx, cfg.y_idx]], wpts_list, cum_dist, prev_i=None
    )

    for _ in range(max_skills):
        if laps_completed >= max_laps or collision_occurred:
            break
        plan_lidar = env.get_lidar()
        plan = planner.plan(plan_lidar, step_idx=plan_call_idx)
        plan_call_idx += 1
        execute_count = min(replan_every_skills, plan["epsilon_seq"].shape[0])
        #predicted_collisions = plan.get("predicted_collisions")
        #if predicted_collisions is not None and np.any(predicted_collisions[:execute_count]):
        #    first_collision = int(np.flatnonzero(predicted_collisions)[0])
        #    print(
        #        "[mpc_race] stopping: the best imagined plan predicts a map "
        #        f"collision at skill {first_collision + 1}."
        #    )
        #    stopped_for_predicted_collision = True
        #    break

        for k in range(execute_count):
            eps_k = plan["epsilon_seq"][k]
            prior_lidar = plan_lidar if k == 0 else plan["predicted_lidar"][k - 1]
            z_k = planner.epsilon_to_z(prior_lidar, eps_k)
            next_state = env.execute_skill(z_k, skill_horizon)
            trajectory.append(next_state.copy())
            executed_zs.append(z_k)
            # I WILL RETURN IT BACK LATER
            #if getattr(env, "crashed", False):
            #    collision_occurred = True
            #    print(f"[mpc_race] stopping: collision detected "
            #          f"(control step {getattr(env, 'crash_step', None)}).")
            #    break

            s, cur_i = calculate_progress_local(
                next_state[[cfg.x_idx, cfg.y_idx]], wpts_list, cum_dist, prev_i=prev_i
            )
            if s < prev_s - total_s * 0.5:  # wrapped past s=0 -> lap completed
                laps_completed += 1
            prev_s = s
            prev_i = cur_i  # carry the index forward for the next step's window search
            print(f"[mpc_race] progress s={s:.3f}/{total_s:.3f}, laps={laps_completed}")

        if laps_completed >= max_laps or collision_occurred:
            break

    return {
        "trajectory_raw": np.stack(trajectory, axis=0),
        "executed_zs": np.stack(executed_zs, axis=0) if executed_zs else np.zeros((0, planner.model.z_dim)),
        "laps_completed": laps_completed,
        "final_progress_s": prev_s,
        "total_track_s": total_s,
        "stopped_for_predicted_collision": stopped_for_predicted_collision,
        "collision": collision_occurred,   
    }


In [ ]:
@dataclass
class KinematicBicycleParams:
    wheelbase_m: float = 0.33
    max_steer_rad: float = 0.4
    max_speed_mps: float = 8.0
    min_speed_mps: float = 0.0
    max_accel_mps2: float = 8.0


def bicycle_rollout(state0: np.ndarray, actions: np.ndarray, dt: float,
                     params: KinematicBicycleParams) -> np.ndarray:
    """Returns (H, 4) array of [x, y, yaw, v] states, one row per action step."""
    x, y, yaw, v = state0
    out = np.zeros((actions.shape[0], 4), dtype=np.float32)
    for h in range(actions.shape[0]):
        steer = float(np.clip(actions[h, 0], -params.max_steer_rad, params.max_steer_rad))
        speed_cmd = float(np.clip(actions[h, 1], params.min_speed_mps, params.max_speed_mps))

        accel = np.clip((speed_cmd - v) / dt, -params.max_accel_mps2, params.max_accel_mps2)
        v = float(np.clip(v + accel * dt, params.min_speed_mps, params.max_speed_mps))
        yaw = yaw + (v / params.wheelbase_m) * np.tan(steer) * dt
        x = x + v * np.cos(yaw) * dt
        y = y + v * np.sin(yaw) * dt

        out[h] = [x, y, yaw, v]
    return out


@dataclass
class MPCConfig:
    horizon: int = 15
    population_size: int = 512
    cem_iters: int = 5
    elite_frac: float = 0.15
    init_steer_std: float = 0.15
    init_speed_std: float = 1.5
    dt: float = 0.05

    @property
    def num_elite(self) -> int:
        return max(1, int(round(self.population_size * self.elite_frac)))


class KinematicMPCController:

    def __init__(self, reward_fn: CenterlineProgressReward, params: KinematicBicycleParams,
                 cfg: MPCConfig, seed: int = 0,
                 classic_action_fn: Optional[Callable[[np.ndarray], np.ndarray]] = None):
        self.reward_fn = reward_fn
        self.params = params
        self.cfg = cfg
        self.rng = np.random.RandomState(seed)
        self.classic_action_fn = classic_action_fn

        if self.reward_fn.reward_mode == "tal" and classic_action_fn is None:
            raise ValueError("reward_mode='tal' requires classic_action_fn to be provided")

    def _state4(self, raw_state: np.ndarray) -> np.ndarray:
        return np.array([raw_state[0], raw_state[1], raw_state[4], raw_state[3]], dtype=np.float32)

    def act(self, raw_state: np.ndarray) -> np.ndarray:
        cfg, params = self.cfg, self.params
        state0 = self._state4(raw_state)

        mean = np.zeros((cfg.horizon, 2), dtype=np.float32)
        mean[:, 1] = min(3.0, params.max_speed_mps)
        std = np.array([cfg.init_steer_std, cfg.init_speed_std], dtype=np.float32)
        std = np.tile(std, (cfg.horizon, 1))

        best_actions, best_reward = mean.copy(), -np.inf

        for _ in range(cfg.cem_iters):
            samples = mean[None] + std[None] * self.rng.normal(
                size=(cfg.population_size, cfg.horizon, 2)
            ).astype(np.float32)

            positions = np.zeros((cfg.population_size, cfg.horizon, 2), dtype=np.float32)
            headings = np.zeros((cfg.population_size, cfg.horizon), dtype=np.float32)
            velocities = np.zeros((cfg.population_size, cfg.horizon), dtype=np.float32)

            for k in range(cfg.population_size):
                traj = bicycle_rollout(state0, samples[k], cfg.dt, params)
                positions[k] = traj[:, 0:2]
                headings[k] = traj[:, 2]
                velocities[k] = traj[:, 3]

            init_position = np.tile(state0[0:2], (cfg.population_size, 1))

            classic_actions = None
            if self.reward_fn.reward_mode == "tal":
                classic_actions = np.zeros_like(samples)
                for k in range(cfg.population_size):
                    for l in range(cfg.horizon):
                        s_l = np.array([positions[k, l, 0], positions[k, l, 1],
                                         headings[k, l], velocities[k, l]], dtype=np.float32)
                        classic_actions[k, l] = self.classic_action_fn(s_l)

            rewards = self.reward_fn(
                positions,
                headings=headings,
                velocities=velocities,
                agent_actions=samples if self.reward_fn.reward_mode == "tal" else None,
                classic_actions=classic_actions,
                collisions=None,
                lap_done=None,
                init_position=init_position,
            )

            elite = np.argsort(-rewards)[: cfg.num_elite]
            mean = samples[elite].mean(axis=0)
            std = samples[elite].std(axis=0).clip(min=1e-3)

            top1 = elite[0]
            if rewards[top1] > best_reward:
                best_reward = float(rewards[top1])
                best_actions = samples[top1].copy()

        return best_actions[0]


def classical_mpc_race(controller: KinematicMPCController, env, waypoints_raw: np.ndarray,
             control_hz: int, max_seconds: float = 60.0, max_laps: int = 1) -> dict:

    trajectory = [env.get_state().copy()]
    wpts_list = waypoints_raw.tolist()

    seg_len = np.linalg.norm(np.diff(waypoints_raw, axis=0), axis=1)
    cum_dist = np.concatenate([[0.0], np.cumsum(seg_len)])
    total_s = float(cum_dist[-1]) + 1e-8

    max_ticks = int(max_seconds * control_hz)
    laps_completed = 0
    collision_occurred = False  
    prev_s = calculate_progress(env.get_state()[0:2], wpts_list, cum_dist)

    for _ in range(max_ticks):
        raw_state = env.get_state()
        action = controller.act(raw_state)
        next_state = env.step_raw_action(action)
        trajectory.append(next_state.copy())

        if getattr(env, "crashed", False):
            collision_occurred = True
            print(f"[classical_mpc_race] stopping: collision detected "
                  f"(control step {getattr(env, 'crash_step', None)}).")
            break

        s = calculate_progress(next_state[0:2], wpts_list, cum_dist)
        if s < prev_s - total_s * 0.5: 
            laps_completed += 1
        prev_s = s

        if laps_completed >= max_laps:
            break

    return {
        "trajectory_raw": np.stack(trajectory, axis=0),
        "laps_completed": laps_completed,
        "final_progress_s": prev_s,
        "total_track_s": total_s,
        "collision": collision_occurred, 
    }

In [ ]:

# --------------------------------------------------------------------------
# LiDAR state representation
# --------------------------------------------------------------------------
# The pose vector (x, y, yaw, ...) is map-specific and not something a real
# car would observe -- it doesn't generalize and leaks map identity into the
# "state". Instead we use the standard LiDAR-beam representation: N evenly
# spaced beams spanning a forward-facing field of view, range-clipped and
# normalized to [0, 1]. Pose is still computed internally (pure pursuit needs
# it to act), it's just no longer part of the state we save for the world
# model.
NUM_BEAMS_OUT = 108
BEAM_FOV_RAD = np.pi          # pi radians, centered on the heading direction
BEAM_MAX_RANGE_M = 10.0       # beams are clipped to this range before scaling
BEAM_COLUMNS = [f"beam_{i:02d}" for i in range(NUM_BEAMS_OUT)]

# f110_gym's default ScanSimulator2D uses 1080 beams over a 270 deg (4.7 rad)
# FOV, symmetric about the heading. We read this from the env if available
# and fall back to these defaults otherwise.
DEFAULT_FULL_NUM_BEAMS = 108
DEFAULT_FULL_FOV_RAD = 4.7


def get_full_scan_geometry(env) -> "tuple[int, float]":
    candidates = [
        lambda: (env.sim.agents[0].scan_simulator.num_beams, env.sim.agents[0].scan_simulator.fov),
        lambda: (env.sim.agents[0].num_beams, env.sim.agents[0].fov),
        lambda: (env.params["num_beams"], env.params["fov"]),
    ]
    for get in candidates:
        try:
            num_beams, fov = get()
            return int(num_beams), float(fov)
        except (AttributeError, KeyError, TypeError):
            continue
    return DEFAULT_FULL_NUM_BEAMS, DEFAULT_FULL_FOV_RAD


def build_beam_sampling_indices(full_num_beams: int, full_fov: float,
                                 out_num_beams: int = NUM_BEAMS_OUT,
                                 out_fov: float = BEAM_FOV_RAD) -> np.ndarray:
    """Precompute, once per env, which indices into the raw full-resolution
    scan correspond to `out_num_beams` evenly spaced beams spanning `out_fov`
    radians centered on the heading direction."""
    full_angles = np.linspace(-full_fov / 2.0, full_fov / 2.0, full_num_beams)
    target_angles = np.linspace(-out_fov / 2.0, out_fov / 2.0, out_num_beams)
    indices = np.array([int(np.argmin(np.abs(full_angles - a))) for a in target_angles])
    return indices


def extract_beams(full_scan: np.ndarray, sample_indices: np.ndarray,
                   max_range: float = BEAM_MAX_RANGE_M) -> np.ndarray:
    """Downsample a raw LiDAR scan and retain ranges in metres.

    ``LidarEncoder`` is the single normalization point shared by training and
    execution.  Saving pre-normalized beams here would make the train-time
    and live-policy inputs differ by a range-scale factor.
    """
    beams = np.asarray(full_scan, dtype=np.float32)[sample_indices]
    beams = np.nan_to_num(beams, nan=max_range, posinf=max_range, neginf=0.0)
    beams = np.clip(beams, 0.0, max_range)
    return beams.astype(np.float32)


def collect_one_run(env, centerline: Centerline, target_speed: float, cfg, rng,
                     num_ticks: int, substeps: int, out_path: str, map_name: str,
                     episode_id: int, randomize_start: bool = True):

    pp_cfg = PurePursuitConfig(
        wheelbase_m=cfg.collection.wheelbase_m, max_steer_rad=cfg.collection.max_steer_rad,
        target_speed_mps=target_speed,
    )
    controller = PurePursuitController(centerline, pp_cfg)

    if randomize_start:
        start_idx = int(rng.randint(0, centerline.num_points))
    else:
        start_idx = 0
    next_idx = (start_idx + 1) % centerline.num_points
    start_xy = centerline.points[start_idx]
    heading = np.arctan2(*(centerline.points[next_idx] - centerline.points[start_idx])[::-1])
    obs, _, done, _ = env.reset(np.array([[start_xy[0], start_xy[1], heading]], dtype=np.float64))

    full_num_beams, full_fov = get_full_scan_geometry(env)
    sample_indices = build_beam_sampling_indices(
        full_num_beams, full_fov, out_num_beams=cfg.data.lidar_dim,
    )
    rows = []
    collided = False
    for step in range(num_ticks):
        try:
            state = np.asarray(env.sim.agents[0].state, dtype=np.float32)
        except AttributeError:
            state = np.array([
                obs["poses_x"][0], obs["poses_y"][0], 0.0,
                obs["linear_vels_x"][0], obs["poses_theta"][0], obs["ang_vels_z"][0], 0.0,
            ], dtype=np.float32)
        scan = np.asarray(obs["scans"][0], dtype=np.float32)  # scan paired with `state` above: both reflect the previous tick's outcome

        x, y, steer_angle, vel, yaw_angle, yaw_rate, slip_angle = state
        action = controller.act(x, y, yaw_angle, vel)
        if cfg.collection.action_noise:
            action = add_exploration_noise(
                action, rng, steer_std=cfg.collection.steer_noise_std,
                speed_std=cfg.collection.speed_noise_std, max_steer_rad=cfg.collection.max_steer_rad,
            )

        for _ in range(substeps):
            obs, _, done, _ = env.step(np.array([action], dtype=np.float64))
            if done:
                break

        collided = bool(done and obs["collisions"][0])
        is_last_tick = step == (num_ticks - 1)
        step_done = bool(done or is_last_tick)

        row = {
            "map_name": map_name, "episode_id": episode_id, "step": step,
            "x": x, "y": y, "steer_angle": steer_angle, "vel": vel,
            "yaw_angle": yaw_angle, "yaw_rate": yaw_rate, "slip_angle": slip_angle,
            "steer_cmd": action[0], "speed_cmd": action[1],
            "done": step_done, "collision": collided,
        }
        beams = extract_beams(scan, sample_indices, max_range=cfg.data.lidar_max_range)
        for i, beam in enumerate(beams):
            row[f"lidar_{i}"] = float(beam)
        rows.append(row)

        if step_done:
            if collided:
                print(f"  [collect] episode {episode_id}: collision -- ended at tick {len(rows)}")
            else:
                print(f"  [collect] episode {episode_id}: finished cleanly at tick {len(rows)}")
            break

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    print(f"  [collect] saved {len(df)} rows -> {out_path}")
    return {"episode_id": episode_id, "num_steps": len(rows), "collision": collided}


def main_generate_offline_data():
    config = "configs/default.yaml"
    cfg = load_config(config)

    rng = np.random.RandomState(cfg.collection.seed)
    substeps = max(1, round(1.0 / (cfg.collection.control_hz * cfg.collection.timestep)))
    run_seconds = cfg.collection.run_seconds
    num_ticks = int(run_seconds * cfg.collection.control_hz)

    # Number of episodes to collect per map. Falls back to laps_per_map for
    # backwards compatibility with older configs.
    num_episodes = getattr(cfg.collection, "episodes_per_map", None) 
    randomize_start = getattr(cfg.collection, "randomize_start", True)

    summary = {}
    for map_name in cfg.collection.map_names:
        map_yaml, map_ext, centerline_csv = discover_map(cfg.data.maps_root, map_name)
        centerline = Centerline.from_csv(centerline_csv)
        out_dir = os.path.join(cfg.collection.out_root, map_name)
        os.makedirs(out_dir, exist_ok=True)

        env = F110Env(map=map_name)

        map_stats = []
        for episode_id in range(num_episodes):
            target_speed = cfg.collection.max_speed_choices[episode_id % len(cfg.collection.max_speed_choices)]
            out_path = os.path.join(out_dir, f"run-{episode_id:04d}_speed{target_speed:.1f}.csv")
            print(f"[collect] map={map_name} episode={episode_id}/{num_episodes - 1} target_speed={target_speed}")
            stats = collect_one_run(env, centerline, target_speed, cfg, rng, num_ticks, substeps,
                                     out_path, map_name, episode_id=episode_id,
                                     randomize_start=randomize_start)
            map_stats.append(stats)

        env.close() if hasattr(env, "close") else None

        n_collisions = sum(1 for s in map_stats if s["collision"])
        total_steps = sum(s["num_steps"] for s in map_stats)
        summary[map_name] = {
            "episodes": len(map_stats), "collisions": n_collisions, "total_steps": total_steps,
        }
        print(f"[collect] map={map_name} done: {len(map_stats)} episodes, "
              f"{n_collisions} collisions, {total_steps} total steps.")

    print("[collect] summary:")
    for map_name, stats in summary.items():
        print(f"  {map_name}: {stats}")
    print("[collect] done.")


In [ ]:
main_generate_offline_data()

In [ ]:
def main_train():
    config = "configs/default.yaml"
    mode = None

    cfg = load_config(config)
    set_seed(cfg.train.seed)

    mode =  cfg.train.mode
    device = resolve_device(cfg.train.device)

    all_runs = discover_runs(cfg.data.root, cfg.data.runs)
    train_names, eval_names = train_test_run_split(all_runs, cfg.data.train_run_frac, seed=cfg.train.seed)
    print(f"[data] {len(all_runs)} runs found (from scripts/generate_offline_data.py). "
          f"train={len(train_names)} eval={len(eval_names)}")

    train_runs, state_norm, action_norm = build_run_arrays(
        root=cfg.data.root, run_names=train_names,
        state_features=cfg.data.state_features, action_features=cfg.data.action_features,
        dropna=cfg.data.dropna, stride=cfg.data.subsample_stride,
    )
    lidar_width = train_runs[0].lidar.shape[1]
    if lidar_width != cfg.data.lidar_dim:
        raise ValueError(
            f"Dataset has {lidar_width} lidar columns but data.lidar_dim is "
            f"{cfg.data.lidar_dim}. Regenerate the offline data with "
            "scripts/generate_offline_data.py; do not train on mixed scan formats."
        )

    os.makedirs(cfg.train.checkpoint_dir, exist_ok=True)
    state_norm.save(os.path.join(cfg.train.checkpoint_dir, "state_normalizer.json"))
    action_norm.save(os.path.join(cfg.train.checkpoint_dir, "action_normalizer.json"))
    with open(os.path.join(cfg.train.checkpoint_dir, "train_eval_split.txt"), "w") as f:
        f.write("TRAIN:\n" + "\n".join(train_names) + "\n\nEVAL:\n" + "\n".join(eval_names) + "\n")

    dataset = OfflineSkillDataset(train_runs, window_len=cfg.data.subtraj_len)
    print(f"[data] {len(dataset)} training sub-trajectory windows "
          f"(T={cfg.data.subtraj_len - 1}, stride={cfg.data.subsample_stride}).")

    dataloader = DataLoader(dataset, batch_size=cfg.train.batch_size, shuffle=True,
                             drop_last=False, num_workers=2, pin_memory=(device == "cuda"))

    if not eval_names:
        raise ValueError("Validation needs at least one held-out run; collect more runs or reduce data.train_run_frac.")
    val_runs, _, _ = build_run_arrays(
        root=cfg.data.root, run_names=eval_names,
        state_features=cfg.data.state_features, action_features=cfg.data.action_features,
        dropna=cfg.data.dropna, stride=cfg.data.subsample_stride,
        state_normalizer=state_norm, action_normalizer=action_norm,
    )
    val_dataset = OfflineSkillDataset(val_runs, window_len=cfg.data.subtraj_len)
    val_dataloader = DataLoader(
        val_dataset, batch_size=cfg.train.batch_size, shuffle=False,
        drop_last=False, num_workers=2, pin_memory=(device == "cuda"),
    )
    print(f"[validation] {len(val_dataset)} held-out windows from {len(eval_names)} runs.")

    model = OPOSMSkillModel(
        state_dim=len(cfg.data.state_features), action_dim=len(cfg.data.action_features),
        z_dim=cfg.model.z_dim, hidden_dim=cfg.model.hidden_dim,
        gru_hidden_dim=cfg.model.gru_hidden_dim, gru_bidirectional=cfg.model.gru_bidirectional,
        min_std=cfg.model.min_std, lidar_dim=cfg.data.lidar_dim,
        obs_embed_dim=cfg.model.obs_embed_dim, lidar_max_range=cfg.data.lidar_max_range,
        tawm_predict_delta=cfg.model.tawm_predict_delta,
    )

    trainer_cfg = EMTrainerConfig(
        lr_e=cfg.train.lr_e, lr_m=cfg.train.lr_m, grad_clip_norm=cfg.train.grad_clip_norm,
        device=device, checkpoint_dir=cfg.train.checkpoint_dir, log_every=cfg.train.log_every,
        save_every_epochs=cfg.train.save_every_epochs, mode=mode,
    )
    trainer = EMTrainer(model, trainer_cfg)

    print(f"[train] mode={mode} device={trainer.device} "
          f"epochs={cfg.train.num_epochs} steps/epoch={cfg.train.steps_per_epoch}")
    trainer.fit(dataloader, num_epochs=cfg.train.num_epochs, steps_per_epoch=cfg.train.steps_per_epoch,
                val_dataloader=val_dataloader)
    print("[train] done.")

In [ ]:
def main_plan_lap(chekpoint, mape , render , env):
    config = "configs/default.yaml"
    cfg = load_config(config)
    out ="outputs/plan_lap"

    device = resolve_device(cfg.planning.device)
    os.makedirs(out, exist_ok=True)

    reward_type = cfg.planning.reward
    state_norm = Normalizer.load(os.path.join(cfg.train.checkpoint_dir, "state_normalizer.json"))
    action_norm = Normalizer.load(os.path.join(cfg.train.checkpoint_dir, "action_normalizer.json"))

    _, _, centerline_csv = discover_map(cfg.data.maps_root, mape)
    centerline = Centerline.from_csv(centerline_csv)
    waypoints_raw = centerline.subsample_waypoints(cfg.planning.waypoint_spacing_m)

    heading0 = np.arctan2(*(centerline.points[1] - centerline.points[0])[::-1])
    initial_state_raw = np.array(
        [centerline.points[0, 0], centerline.points[0, 1], 0.0, 0.0, heading0, 0.0, 0.0],
        dtype=np.float32,
    )

    ckpt = torch.load(chekpoint, map_location=device)
    model = OPOSMSkillModel(
        state_dim=ckpt.get("state_dim", len(cfg.data.state_features)),
        action_dim=ckpt.get("action_dim", len(cfg.data.action_features)),
        z_dim=ckpt.get("z_dim", cfg.model.z_dim),
        lidar_dim=ckpt.get("lidar_dim", cfg.data.lidar_dim),
        hidden_dim=cfg.model.hidden_dim,
        gru_hidden_dim=cfg.model.gru_hidden_dim, gru_bidirectional=cfg.model.gru_bidirectional,
        min_std=cfg.model.min_std, obs_embed_dim=ckpt.get("obs_embed_dim", cfg.model.obs_embed_dim),
        lidar_max_range=cfg.data.lidar_max_range,
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    reward_fn = LidarStructuralReward(
        lidar_dim=model.lidar_dim,
        forward_weight=cfg.planning.forward_reward_weight,
        symmetry_weight=cfg.planning.symmetry_penalty_weight,
        collision_penalty=cfg.planning.lidar_collision_penalty,
        safe_distance=cfg.planning.safe_distance_m,
        forward_fraction=cfg.planning.forward_beam_fraction,
    )
    planner_cfg = CEMPlannerConfig(
        plan_length=cfg.planning.plan_length, population_size=cfg.planning.population_size,
        elite_frac=cfg.planning.elite_frac, cem_iters=cfg.planning.cem_iters,
        init_std=cfg.planning.init_std, device=device,
    )
    # TODO : implment TAL reward function  baseline
    planner = CEMPlanner(model, reward_fn, planner_cfg)
    skill_horizon = cfg.data.subtraj_len - 1

    if env == "tawm":
        env = TAWMRolloutEnvironment(model, state_norm, initial_state_raw, device=device)
    elif env == "kinematic":
        env = KinematicSimEnvironment(model, action_norm, state_norm, initial_state_raw, device=device)
    else:
        map_yaml, map_ext, _ = discover_map(cfg.data.maps_root, mape)
        env_cfg = F1TenthEnvConfig(
            map_yaml=map_yaml, map_ext=map_ext, timestep=cfg.f1tenth_env.timestep,
            control_hz=cfg.f1tenth_env.control_hz, max_steer_rad=cfg.f1tenth_env.max_steer_rad,
            max_speed_mps=cfg.f1tenth_env.max_speed_mps,
            render=cfg.f1tenth_env.render if render is None else render,
            lidar_dim=model.lidar_dim,
            action_repeat=cfg.data.subsample_stride,
        )
        env = F1TenthEnvironment(model, state_norm, action_norm, env_cfg, initial_state_raw, device=device)

    print(f"[plan] map={mape} num_waypoints={waypoints_raw.shape[0]} env={env}")
    try:
        result = mpc_race(
            planner, env, waypoints_raw, skill_horizon=skill_horizon,
            max_skills=cfg.planning.max_skills_per_episode, max_laps=cfg.planning.max_laps,
            replan_every_skills=cfg.planning.replan_every_skills,
        )
    finally:
        if hasattr(env, "close"):
            env.close()

    print(f"[plan]({result['laps_completed']} laps) in {result['executed_zs'].shape[0]} skills.")
    if getattr(env, "crashed", False):
        print(f"[plan] WARNING: collision during execution (control step {env.crash_step}).")

    traj = result["trajectory_raw"]
    np.savez(os.path.join(out, f"{mape}_plan.npz"),
              trajectory=traj, executed_zs=result["executed_zs"], waypoints=waypoints_raw)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(centerline.points[:, 0], centerline.points[:, 1], color="lightgray",
            linewidth=1, label="Centerline")
    ax.plot(traj[:, 0], traj[:, 1], color="crimson", linewidth=2, marker="o",
            markersize=3, label=f"OPOSM plan ({env} execution)")
    ax.scatter(waypoints_raw[:, 0], waypoints_raw[:, 1], color="gold", edgecolor="black",
               s=60, marker="X", zorder=5, label="Waypoints")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(f"OPOSM zero-shot lap plan: {mape}")
    ax.legend()
    ax.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    out_png = os.path.join(out, f"{mape}_plan.png")
    fig.savefig(out_png, dpi=150)
    print(f"[plan] saved trajectory plot -> {out_png}")

In [ ]:

@dataclass
class F1TenthEnvConfig:
    map_yaml: str
    map_ext: str
    timestep: float = 0.01
    control_hz: int = 20
    max_steer_rad: float = 0.4
    max_speed_mps: float = 8.0
    render: bool = False
    # Number of beams fed to the learned policy.  ``None`` uses the width the
    # loaded model was trained with.  Do not use a fixed stride here: F1TENTH
    # installations can expose different native scan resolutions.
    lidar_dim: Optional[int] = None
    action_repeat: int = 1
def load_model(cfg, checkpoint_path, device):
    model = OPOSMSkillModel(
        action_dim=len(cfg.data.action_features),
        z_dim=cfg.model.z_dim, hidden_dim=cfg.model.hidden_dim,
        gru_hidden_dim=cfg.model.gru_hidden_dim, gru_bidirectional=cfg.model.gru_bidirectional,
        min_std=cfg.model.min_std,
        tawm_predict_delta=cfg.model.tawm_predict_delta , state_dim=len(cfg.data.state_features) , lidar_dim=cfg.data.lidar_dim
    )
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()
    return model


@torch.no_grad()
def tawm_prediction_mse(model, dataloader, device, num_batches=None):
    mses, elbos = [], []
    for i, batch in enumerate(dataloader):
        if num_batches is not None and i >= num_batches:
            break
        batch = {k: v.to(device) for k, v in batch.items()}
        terms = model.elbo_terms(batch)
        o0_emb = model.lidar_encoder(batch["lidar_seq"][:, 0])
        mu, _ = model.tawm(o0_emb, terms["z"])
        target = torch.clamp(batch["lidar_T"], min=0.0, max=model.lidar_max_range) / model.lidar_max_range
        mses.append(((mu - target) ** 2).mean().item()) 
        elbos.append(terms["elbo"].mean().item())
    return float(np.mean(mses)), float(np.mean(elbos))


def _summarize_runs(results_for_all_runs):

    track_coverages = [r["final_progress_s"] / r["total_track_s"] for r in results_for_all_runs]
    track_coverage = float(np.mean(track_coverages))
    lapc_completer = float(np.mean([r["laps_completed"] for r in results_for_all_runs]))
    collision_rate = float(np.mean([bool(r.get("collision", False)) for r in results_for_all_runs]))
    return track_coverage, lapc_completer, collision_rate





def oposm_planning_eval(model, action_norm, state_norm, cfg, device, eval_runs, maps_root, num_runs=1):
    results_for_all_runs = []
    for run in eval_runs[:num_runs]:
        map_yaml , map_ext , centerline = load_centerline_for_run(maps_root, run)
        waypoints = centerline.subsample_waypoints(cfg.planning.waypoint_spacing_m)
        reward_fn = LidarStructuralReward(
            lidar_dim=model.lidar_dim,
            forward_weight=cfg.planning.forward_reward_weight,
            symmetry_weight=cfg.planning.symmetry_penalty_weight,
            collision_penalty=cfg.planning.lidar_collision_penalty,
            safe_distance=cfg.planning.safe_distance_m,
            forward_fraction=cfg.planning.forward_beam_fraction,
        )
        skill_horizon = cfg.data.subtraj_len - 1
        planner_cfg = CEMPlannerConfig(
            plan_length=cfg.planning.plan_length, population_size=cfg.planning.population_size,
            elite_frac=cfg.planning.elite_frac, cem_iters=cfg.planning.cem_iters,
            init_std=cfg.planning.init_std, device=device,
        )
        planner = CEMPlanner(model, reward_fn, planner_cfg)
        env_cfg = F1TenthEnvConfig(
                    map_yaml=map_yaml,
                    map_ext=map_ext,
                    lidar_dim=model.lidar_dim,
                    action_repeat=cfg.data.subsample_stride,
                )
        heading0 = np.arctan2(*(centerline.points[1] - centerline.points[0])[::-1])
        initial_state_raw = np.array(
        [centerline.points[0, 0], centerline.points[0, 1], 0.0, 0.0, heading0, 0.0, 0.0],
        dtype=np.float32,
    )
        env = F1TenthEnvironment(model, state_norm, action_norm, cfg=env_cfg,
                                  initial_state_raw=initial_state_raw, device=device)
        result = mpc_race(
            planner, env, waypoints, skill_horizon=skill_horizon,
            max_skills=cfg.planning.max_skills_per_episode, max_laps=cfg.planning.max_laps,
            replan_every_skills=cfg.planning.replan_every_skills,
        )
        results_for_all_runs.append(result)

    track_coverage, lapc_completer, collision_rate = _summarize_runs(results_for_all_runs)

    return results_for_all_runs, track_coverage, lapc_completer, collision_rate


def classical_mpc_eval(cfg, eval_runs, maps_root, num_runs=1):
    results_for_all_runs = []
    for run in eval_runs[:num_runs]:
        map_yaml , map_ext , centerline = load_centerline_for_run(maps_root, run)
        waypoints = centerline.subsample_waypoints(cfg.planning.waypoint_spacing_m)
        reward_fn = CenterlineProgressReward(
            waypoints,
            waypoint_bonus=cfg.planning.waypoint_bonus_reward,
            progress_weight=cfg.planning.progress_reward_weight,
            off_track_penalty=cfg.planning.off_track_penalty,
        )
        mpc_cfg = MPCConfig(
            horizon=cfg.mpc.horizon, population_size=cfg.mpc.population_size,
            cem_iters=cfg.mpc.cem_iters, elite_frac=cfg.mpc.elite_frac,
            init_steer_std=cfg.mpc.init_steer_std, init_speed_std=cfg.mpc.init_speed_std,
            dt=cfg.mpc.dt,
        )
        params = KinematicBicycleParams(
            wheelbase_m=cfg.collection.wheelbase_m, max_steer_rad=cfg.collection.max_steer_rad,
        )
        controller = KinematicMPCController(reward_fn, params, mpc_cfg)
        env_cfg = F1TenthEnvConfig(
            map_yaml=map_yaml,
            map_ext=map_ext
        ) 
        heading0 = np.arctan2(*(centerline.points[1] - centerline.points[0])[::-1])
        initial_state_raw = np.array(
        [centerline.points[0, 0], centerline.points[0, 1], 0.0, 0.0, heading0, 0.0, 0.0],
        dtype=np.float32,
    )

        env = F1TenthEnvironment(model=None, state_normalizer=None, action_normalizer=None,
                                  cfg=env_cfg, initial_state_raw=initial_state_raw)
        result = classical_mpc_race(controller, env, waypoints, control_hz=int(1 / mpc_cfg.dt),
                                     max_seconds=cfg.mpc.max_seconds, max_laps=cfg.planning.max_laps)
        results_for_all_runs.append(result)

    track_coverage, lapc_completer, collision_rate = _summarize_runs(results_for_all_runs)

    return results_for_all_runs, track_coverage, lapc_completer, collision_rate


def main_evaluate(em_checkpoint  ,normalizer_dir , num_planning_runs=3 , skip_mpc_baseline="do" , output_dir="resutls" ):
    config = "configs/default.yaml"

    cfg = load_config(config)
    device = resolve_device(cfg.train.device)

    state_norm = Normalizer.load(os.path.join(normalizer_dir, "state_normalizer.json"))
    action_norm = Normalizer.load(os.path.join(normalizer_dir, "action_normalizer.json"))
    all_runs = discover_runs(cfg.data.root, cfg.data.runs)
    train_names, eval_names = train_test_run_split(all_runs, cfg.data.train_run_frac, seed=cfg.train.seed)
    #eval_runs, _, _, _ = build_run_arrays(
    #    root=cfg.data.root, run_names=eval_names,state_features=cfg.data.state_features , 
    #     action_features=cfg.data.action_features,
    #    dropna=cfg.data.dropna, stride=cfg.data.subsample_stride,
    #    state_normalizer=state_norm, action_normalizer=action_norm,
    #)
    eval_runs = ["Melbourne" , "Monza" , "Shanghai"]
    results = {}
    for label, ckpt_path in [
        ("EM (ours)", em_checkpoint),
    ]:
        print(f"\n=== Evaluating: {label} ===")

        model = load_model(cfg, ckpt_path, device)

        results_for_all_runs, track_coverage, lapc_completer, collision_rate = oposm_planning_eval(
            model,
            action_norm,
            state_norm,
            cfg,
            device,
            eval_runs,
            cfg.data.maps_root,
            num_runs=num_planning_runs
        )
        results[label] = {
            "planning": {
                "track_coverage": float(track_coverage),
                "laps_completed": float(lapc_completer),
                "collision_rate": float(collision_rate),   # NEW: fraction of runs that crashed
                "num_runs": len(results_for_all_runs),

                "runs": [
                    {
                        "laps_completed": int(run["laps_completed"]),
                        "final_progress_s": float(run["final_progress_s"]),
                        "total_track_s": float(run["total_track_s"]),
                        "collision": bool(run.get("collision", False)),  # NEW
                    }
                    for run in results_for_all_runs
                ],
            }
        }


    # ============================================================
    # Evaluate classical MPC baseline
    # ============================================================

    if not skip_mpc_baseline:

        print("\n=== Evaluating: Kinematic MPC baseline (classical control) ===")

        results_for_all_runs, track_coverage, lapc_completer, collision_rate = classical_mpc_eval(
            cfg,
            eval_runs,
            cfg.data.maps_root,
            num_runs=num_planning_runs
        )

        results["Kinematic MPC"] = {
            "prediction": None,

            "planning": {
                "track_coverage": float(track_coverage),
                "laps_completed": float(lapc_completer),
                "collision_rate": float(collision_rate),   # NEW: fraction of runs that crashed
                "num_runs": len(results_for_all_runs),

                "runs": [
                    {
                        "laps_completed": int(run["laps_completed"]),
                        "final_progress_s": float(run["final_progress_s"]),
                        "total_track_s": float(run["total_track_s"]),
                        "collision": bool(run.get("collision", False)),  # NEW
                    }
                    for run in results_for_all_runs
                ],
            }
        }


    # ============================================================
    # Save results
    # ============================================================

    os.makedirs(output_dir, exist_ok=True)

    results_path = os.path.join(
        output_dir,
        "evaluation_results.json"
    )

    with open(results_path, "w") as f:
        json.dump(results, f, indent=4)

    print(f"\nResults saved to: {results_path}")

In [ ]:
main_evaluate("checkpoints/oposm_final.pt" ,"checkpoints" )

In [ ]:
main_train()

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np


# --------------------------------------------------------------------------
# Edit these and re-run the cell
# --------------------------------------------------------------------------
METRICS_PATH = "checkpoints/metrics.json"   # path to metrics.json written by EMTrainer
OUT_PATH = "figures/training_curves"        # no extension; saved as .png and .pdf
SMOOTH = 15                                 # moving-average window in steps, 0 disables


# --------------------------------------------------------------------------
# Semantic color palette (figures4papers convention)
# --------------------------------------------------------------------------
PALETTE = {
    "blue_main": "#0F4D92",
    "blue_secondary": "#3775BA",
    "green_1": "#DDF3DE", "green_2": "#AADCA9", "green_3": "#8BCF8B",
    "red_1": "#F6CFCB", "red_2": "#E9A6A1", "red_strong": "#B64342",
    "neutral": "#CFCECE", "highlight": "#FFD700",
    "teal": "#42949E", "violet": "#9A4D8E",
}
DEFAULT_COLORS = [
    PALETTE["blue_main"], PALETTE["green_3"], PALETTE["red_strong"],
    PALETTE["teal"], PALETTE["violet"], PALETTE["neutral"],
]


@dataclass(frozen=True)
class FigureStyle:
    font_size: int = 16
    axes_linewidth: float = 2.0
    use_tex: bool = False
    font_family: tuple = ("DejaVu Sans", "Helvetica", "Arial", "sans-serif")


def apply_publication_style(style: Optional[FigureStyle] = None) -> None:
    """Configure matplotlib rcParams once, before creating any figure."""
    style = style or FigureStyle()
    plt.rcParams.update({
        "font.size": style.font_size,
        "font.family": "sans-serif",
        "font.sans-serif": list(style.font_family),
        "axes.linewidth": style.axes_linewidth,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelsize": style.font_size,
        "xtick.labelsize": style.font_size - 2,
        "ytick.labelsize": style.font_size - 2,
        "xtick.major.width": style.axes_linewidth * 0.8,
        "ytick.major.width": style.axes_linewidth * 0.8,
        "legend.frameon": False,
        "legend.fontsize": style.font_size - 2,
        "text.usetex": style.use_tex,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,   # keep text as text, not curves, in vector exports
        "ps.fonttype": 42,
    })


def create_subplots(nrows: int = 1, ncols: int = 1, figsize: Optional[tuple] = None, **kwargs):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kwargs)
    axes = np.atleast_1d(axes).ravel()
    return fig, axes


def make_trend(
    ax, x: Sequence[float], y_series: Sequence[Sequence[float]], labels: Sequence[str],
    colors: Optional[Sequence[str]] = None, ylabel: Optional[str] = None,
    xlabel: Optional[str] = None, show_shadow: bool = False,
):
    """Plot one or more line series with the shared publication style."""
    x = np.asarray(x)
    colors = colors or DEFAULT_COLORS
    for i, (y, label) in enumerate(zip(y_series, labels)):
        y = np.asarray(y)
        color = colors[i % len(colors)]
        ax.plot(x, y, color=color, linewidth=2.2, label=label, solid_capstyle="round")
        if show_shadow:
            # Lightweight "uncertainty" band: local rolling std, purely visual.
            win = max(3, len(y) // 50)
            roll_std = _rolling_std(y, win)
            ax.fill_between(x, y - roll_std, y + roll_std, color=color, alpha=0.15, linewidth=0)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if len(labels) > 1:
        ax.legend(loc="best")
    ax.grid(alpha=0.25, linewidth=0.8)
    return ax


def finalize_figure(
    fig, out_path: str, formats: Optional[Iterable[str]] = None,
    dpi: int = 300, close: bool = True, pad: float = 0.05, **kwargs,
):
    """Export fig to one or more formats, creating parent dirs as needed."""
    valid_formats = {"pdf", "svg", "eps", "png", "jpg", "jpeg", "tif", "tiff"}
    out_path = Path(out_path)
    if formats is None:
        ext = out_path.suffix.lstrip(".")
        formats = [ext] if ext in valid_formats else ["png", "pdf"]
        out_path = out_path.with_suffix("")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    saved = []
    for fmt in formats:
        if fmt not in valid_formats:
            raise ValueError(f"Unsupported format '{fmt}'. Must be one of {sorted(valid_formats)}")
        p = out_path.with_suffix(f".{fmt}")
        fig.savefig(p, format=fmt, dpi=dpi, bbox_inches="tight", pad_inches=pad, **kwargs)
        saved.append(p)
    if close:
        plt.close(fig)
    return saved


def _rolling_std(y: np.ndarray, window: int) -> np.ndarray:
    if window <= 1 or len(y) < window:
        return np.zeros_like(y)
    mean = _smooth(y, window)
    mean_sq = _smooth(y ** 2, window)
    var = np.clip(mean_sq - mean ** 2, 0, None)
    return np.sqrt(var)


def _smooth(y: np.ndarray, window: int) -> np.ndarray:
    """Centered moving-average smoothing for noisy per-step losses.

    Pads with edge values before convolving so the first/last ~window/2
    points aren't pulled toward zero by implicit zero-padding (the usual
    artifact of naive mode='same' convolution).
    """
    if window <= 1 or len(y) < window:
        return y
    half = window // 2
    padded = np.pad(y, (half, window - 1 - half), mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(padded, kernel, mode="valid")


# --------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------
def load_metrics(path: str) -> dict:
    with open(path, "r") as f:
        records = json.load(f)
    if not records:
        raise ValueError(f"'{path}' contains no records.")

    keys = set()
    for r in records:
        keys.update(r.keys())
    data = {k: [] for k in keys}
    for r in records:
        for k in keys:
            data[k].append(r.get(k, np.nan))
    return {k: np.asarray(v, dtype=float) for k, v in data.items()}


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------
def plot_training_curves(metrics_path: str = METRICS_PATH, out_path: str = OUT_PATH, smooth: int = SMOOTH):
    """Load metrics_path and render/save the loss + ELBO panels. Returns the fig for inline display."""
    data = load_metrics(metrics_path)
    steps = data["step"]

    apply_publication_style(FigureStyle(font_size=16, axes_linewidth=2.0))
    fig, axes = create_subplots(1, 2, figsize=(11, 4.2))

    # --- Left panel: training loss(es) ---
    if "e_loss" in data and "m_loss" in data:
        loss_series = [data["e_loss"], data["m_loss"]]
        loss_labels = ["E-step loss", "M-step loss"]
    else:
        loss_series = [data["joint_loss"]]
        loss_labels = ["Joint loss"]

    if smooth > 1:
        loss_series = [_smooth(y, smooth) for y in loss_series]

    make_trend(
        axes[0], steps, loss_series, loss_labels,
        colors=[PALETTE["blue_main"], PALETTE["red_strong"]],
        xlabel="Training step", ylabel="Loss",
    )
    axes[0].set_title("Training loss", fontsize=16, pad=10)

    # --- Right panel: ELBO ---
    elbo = data["elbo"]
    if smooth > 1:
        elbo = _smooth(elbo, smooth)
    make_trend(
        axes[1], steps, [elbo], ["ELBO"],
        colors=[PALETTE["teal"]],
        xlabel="Training step", ylabel="ELBO",
    )
    axes[1].set_title("Evidence lower bound", fontsize=16, pad=10)

    fig.tight_layout()
    saved = finalize_figure(fig, out_path, formats=["png", "pdf"], dpi=300, close=False)
    print(f"[plot] saved: {', '.join(str(p) for p in saved)}")
    return fig


# Run directly (e.g. as a notebook cell): produces + displays the figure.
fig = plot_training_curves(METRICS_PATH, OUT_PATH, SMOOTH)
fig